# Giai đoạn 3B: Chuẩn hoá lại cải tiến v2 → `GridMultiVectorAdapter` (v3)
Mỗi sửa đổi ứng với một vấn đề trong `danh_gia_cai_tien_stage2_v2.md`:

| # | Vấn đề ở v2 | Sửa ở v3 |
|---|---|---|
| B1 | Query không có tiền tố `'Query: '` (sai giao thức train của ColQwen2.5-v0.2) | Encode lại query với tiền tố; báo cáo cả mốc cũ để đối chiếu |
| B2 | Cửa sổ 1D 8 token lệch ~4 token prompt, vắt qua hàng ảnh | Cửa sổ **2×4 trên lưới ảnh thật** (H'×W', raster); token prompt gộp riêng |
| B3 | Anchor cố định ở vị trí 0 và 4 của cửa sổ | Anchor = token saliency cao nhất + token xa nó nhất (bao phủ), gán token về anchor gần nhất |
| B4 | GroupNorm tính trên cả padding → output phụ thuộc batch | LayerNorm theo token + conv 2D trên vùng mask; **unit test bất biến batch** |
| B5 | Gate "position-free" vẫn lộ vị trí (f2/f3 = 0 ở biên), áp cả lên tiền tố/augmentation | f2/f3 biên = f4; gate chỉ trên token nội dung; phạt tương quan vị trí–gate; phân tích trên toàn bộ test |
| B6 | "Strong-Teacher KD" thực ra là teacher yếu hơn student, gradient nhỏ ~67× | Teacher = V3-Full cùng seed (mạnh hơn), cùng nhiệt độ với CE, không nhân τ² |
| B7 | Nhánh Mix đọc cột `image` (ảnh tiếng Anh gốc) của 5CD-AI | Dùng `vi_image`; thêm nhánh **EN replay** chống quên tiếng Anh |
| A1–A4 | Thiếu baseline cùng ngân sách, thiếu ablation, test cùng miền | Hier **cùng ngân sách** với v3, ablation từng khối (3 seed), OOD ViInfographicVQA, nửa test "xa train" |
| C1–C5 | p = 0.000, không Holm, không kiểm định EN/slice, so thời gian khác thiết bị | Randomization test + Holm, kiểm định EN/OOD, CI cho slice, đo thời gian **cả GPU và CPU** |

Chạy **sau** `stage3_finetune_baselines.ipynb` để bảng cuối có đủ các dòng `FT-*` (nếu chưa chạy, bảng vẫn sinh nhưng thiếu các dòng đó).

In [ ]:
!pip install -q "colpali-engine==0.3.18" scipy pytesseract rank_bm25 timm einops matplotlib
!pip uninstall -y -q torchao
!pip install -q eva-decord || pip install -q decord || true

In [ ]:
import os, gc, glob, json, time, random, hashlib, tarfile, re, subprocess, shutil, copy, traceback
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, login
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from transformers import get_cosine_schedule_with_warmup

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---- Dán HuggingFace Token trực tiếp tại đây nếu chạy nhánh 5CD-AI (NHỚ XÓA TOKEN TRƯỚC KHI NỘP / PUSH GITHUB) ----
HF_TOKEN = ""

# ---- Thư mục lưu trữ dùng chung cache Giai đoạn 1 (ghi kết quả Giai đoạn 2 ra file JSON riêng) ----
_colab_secret_tok = ""
if os.path.isdir("/kaggle/working"):
    SAVE_DIR, DATA_DIR = "/kaggle/working/ViColPali_Stage1_Fresh", "/tmp/vdoc"
elif os.path.exists("/var/colab/hostname"):
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    SAVE_DIR, DATA_DIR = "/content/drive/MyDrive/ViColPali_Stage1_Fresh", "/content/vdoc"
    try:
        _colab_secret_tok = userdata.get("HF_TOKEN") or ""
    except Exception:
        _colab_secret_tok = ""
else:
    SAVE_DIR, DATA_DIR = "./ViColPali_Stage1_Fresh", "./vdoc"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

_hf_tok = (HF_TOKEN or _colab_secret_tok or os.environ.get("HF_TOKEN", "")).strip()
if _hf_tok:
    os.environ["HF_TOKEN"] = _hf_tok
    login(token=_hf_tok, add_to_git_credential=False)
    print("✓ Đã xác thực HuggingFace bằng HF_TOKEN.")
else:
    print("Chưa cung cấp HF_TOKEN (V-Doc/ColQwen tải public bình thường; nhánh 5CD-AI gated cần HF_TOKEN).")

# ---- Hằng số chia dữ liệu (giữ nguyên 100% với best.ipynb & Giai đoạn 1) ----
SEED = 42
N_SHARDS = 1
BLOCK, TEST_EVERY = 200, 10
MAX_TEST = 1500
DATA_SIZES = [1000]
N_TRAIN_PAGES, MAX_Q_PER_PAGE, N_VAL = 2000, 2, 300

# ---- Tham số huấn luyện (GIỮ NGUYÊN Giai đoạn 2 để so sánh công bằng) ----
SEEDS = [0, 1, 2]
EPOCHS = 12
WARMUP_EPOCHS = 1
LR_ADAPTER = 5e-4
WEIGHT_DECAY = 2e-2
BQ = 32
HARD_NEG = 7
TEMP_CE = 0.10
TEMP_KD = 0.15
ANCHOR_W = 0.05
SCORE_CHUNK = 8
DOC_CHUNK = 64

# ---- Cấu hình trộn dữ liệu đa miền sạch 5CD-AI ----
RUN_5CD_MIX = True
N_5CD_PAGES = 1500
MAX_PER_5CD_SOURCE = 500
MAX_5CD_SCAN = 25000

# ---- Công tắc chạy từng phần ----
RUN_COLVINTERN = False  # ColVintern-1B-v1 custom code không tương thích transformers 5.x trên Colab mới ('all_tied_weights_keys')
RUN_HIER_QWEN = True
RUN_QWEN_ADAPTERS = True
RUN_GATE_VIS = True
RUN_EN_CHECK = True
RUN_SLICE_ANALYSIS = True
RUN_PLOT = True
PREFER_FP16_ON_T4 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
VINTERN_IMG_BS = 8 if VRAM_GB >= 22 else 4
QWEN_IMG_BS = 4 if VRAM_GB >= 22 else 2

STAGE1_RES_PATH = f"{SAVE_DIR}/results_s{N_SHARDS}_b{BLOCK}_t{TEST_EVERY}_m{MAX_TEST}.json"

print("SAVE_DIR =", SAVE_DIR, "| DATA_DIR =", DATA_DIR)
if torch.cuda.is_available():
    print(
        f"GPU: {torch.cuda.get_device_name()} ({VRAM_GB:.1f} GB VRAM) | "
        f"bf16 supported: {torch.cuda.is_bf16_supported()} | "
        f"batch_size (Vintern/Qwen): {VINTERN_IMG_BS}/{QWEN_IMG_BS}"
    )
else:
    print("Đang chạy trên CPU")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
# =====================================================================================
# ---- Bổ sung Giai đoạn 3 (dùng chung cho 2 notebook stage3_*) ----
# =====================================================================================
# (1) Sửa giao thức query: colpali-engine >= 0.3.13 KHÔNG còn tự thêm tiền tố "Query: " (query_prefix = ""),
#     trong khi vidore/colqwen2.5-v0.2 được huấn luyện với "Query: " + text + 10×<|endoftext|> (xem model card).
QUERY_PREFIX = "Query: "
QWEN_ID = "vidore/colqwen2.5-v0.2"

# (2) Bộ test tiếng Việt NGOÀI MIỀN: ViInfographicVQA (single-image test: 193 ảnh / 1.374 câu hỏi người kiểm duyệt)
RUN_OOD = True
OOD_REPO = "duytranus/ViInfographicVQA"
OOD_MAX_Q = 1374

# (3) Thống kê: randomization test có cặp (Smucker et al., 2007) + hiệu chỉnh Holm; p = (b+1)/(B+1), không bao giờ in 0
N_PERM = 50000
N_BOOT = 2000

# ---- Đường dẫn cache dùng chung (tái sử dụng nguyên cache Giai đoạn 2, KHÔNG ghi đè) ----
QWEN_TEST_CACHE = f"{SAVE_DIR}/colqwen25_v02_test_emb.pt"
QWEN_TRVAL_CACHE = f"{SAVE_DIR}/colqwen25_v02_trval_s{N_SHARDS}_tr{N_TRAIN_PAGES}x{MAX_Q_PER_PAGE}_v{N_VAL}.pt"
QPREFIX_CACHE = f"{SAVE_DIR}/stage3_colqwen25_v02_queries_prefix_s{N_SHARDS}_tr{N_TRAIN_PAGES}x{MAX_Q_PER_PAGE}_v{N_VAL}.pt"
OOD_CACHE = f"{SAVE_DIR}/stage3_colqwen25_v02_ood_viinfographic_q{OOD_MAX_Q}.pt"
HIER_CACHE = f"{SAVE_DIR}/stage3_colqwen25_v02_hier4_s{N_SHARDS}_tr{N_TRAIN_PAGES}x{MAX_Q_PER_PAGE}_v{N_VAL}.pt"
RES_V2_PATH = f"{SAVE_DIR}/results_stage2_s{N_SHARDS}_b{BLOCK}_t{TEST_EVERY}_m{MAX_TEST}.json"
RES_A_PATH = f"{SAVE_DIR}/results_stage3_baselines_s{N_SHARDS}_b{BLOCK}_t{TEST_EVERY}_m{MAX_TEST}.json"
RES_B_PATH = f"{SAVE_DIR}/results_stage3_v3_s{N_SHARDS}_b{BLOCK}_t{TEST_EVERY}_m{MAX_TEST}.json"

# ---- Công tắc riêng notebook V3 ----
NEED_HIER_TRAIN = False
RUN_V3 = True                   # V3-A, V3-Full, V3-C, V3-C −KD (3 seed)
RUN_ABLATIONS = True            # bỏ gate / bỏ subword / bỏ doc adapter (3 seed mỗi bản)
RUN_V2_RERUN = True             # kiến trúc v2 nguyên bản + 'Query:' (đối chứng)
RUN_EN_REPLAY = True            # trộn 1.000 trang tiếng Anh vidore/colpali_train_set để chống quên EN
N_REPLAY = 1000
RUN_5CD_VI_MIX = True           # 5CD-AI với cột vi_image (tự bỏ qua nếu không có HF_TOKEN)
KD_W_V3 = 1.0                   # KD từ V3-Full, cùng nhiệt độ TEMP_CE, không nhân τ²
POS_W_V3 = 0.1                  # phạt tương quan vị trí–gate
RES_PATH = RES_B_PATH
print("RES_PATH (notebook này) =", RES_PATH)

## 1. Tải `V-Doc` + `EN DocVQA` & băm MD5 (GIỮ NGUYÊN 100% Giai đoạn 2)
Cùng hằng số chia (`BLOCK = 200`, `TEST_EVERY = 10`, `SEED = 42`) → cùng 2.000 trang train / 300 val / 1.067 test / 500 EN như Giai đoạn 2, nên mọi so sánh có cặp (paired) theo từng câu hỏi là hợp lệ.

In [ ]:
REPO = "trannhiem/TranNhiem-Vietnamese-DocumentImage-Reasoning"
jsonl = hf_hub_download(REPO, "data/vdoc.jsonl", repo_type="dataset", local_dir=DATA_DIR)
for s in range(N_SHARDS):
    marker = f"{DATA_DIR}/shards/images-{s:05d}.done"
    if os.path.exists(marker):
        continue
    tar_path = hf_hub_download(REPO, f"shards/images-{s:05d}.tar", repo_type="dataset", local_dir=DATA_DIR)
    with tarfile.open(tar_path) as t:
        t.extractall(DATA_DIR, filter="data")
    os.remove(tar_path)
    open(marker, "w").close()

records, skipped = [], 0
with open(jsonl, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        path = os.path.join(DATA_DIR, r["image"])
        if not os.path.exists(path):
            continue
        if not str(r["id"]).isdigit():
            skipped += 1
            continue
        qs = [t["value"].strip() for t in r["conversations"]
              if t["from"] == "human" and isinstance(t["value"], str) and len(t["value"].strip()) > 10]
        if qs:
            records.append({"id": int(r["id"]), "path": path, "qs": qs})
print(f"{len(records)} trang V-Doc hợp lệ (bỏ {skipped} trang có id không phải số)")

def img_hash_pil(im):
    return hashlib.md5(im.convert("L").resize((32, 32)).tobytes()).hexdigest()

def img_hash(p):
    with Image.open(p) as im:
        im.draft("L", (64, 64))
        return img_hash_pil(im)

for r in tqdm(records, desc="hash ảnh V-Doc"):
    r["h"] = img_hash(r["path"])

records.sort(key=lambda r: r["id"])
is_test = lambda r: (r["id"] // BLOCK) % TEST_EVERY == 0
test_pages, test_h = [], set()
for r in records:
    if is_test(r) and r["h"] not in test_h:
        test_h.add(r["h"])
        test_pages.append(r)
random.Random(SEED).shuffle(test_pages)
test_pages = test_pages[:MAX_TEST]
test_h = {r["h"] for r in test_pages}
train_pages = [r for r in records if not is_test(r) and r["h"] not in test_h]

TEST_DOCS = [r["path"] for r in test_pages]
TEST_Q = [r["qs"][0] for r in test_pages]
TEST_T = list(range(len(test_pages)))

rng = random.Random(SEED)
is_val = lambda r: (r["id"] // BLOCK) % TEST_EVERY == 1
val_pages, val_h = [], set()
for r in train_pages:
    if is_val(r) and r["h"] not in val_h:
        val_h.add(r["h"])
        val_pages.append(r)
rng.shuffle(val_pages)
val_pages = val_pages[:N_VAL]
val_h = {r["h"] for r in val_pages}

sg_pages = [r for r in train_pages if not is_val(r) and r["h"] not in val_h]
rng.shuffle(sg_pages)
sg_pages = sg_pages[:N_TRAIN_PAGES]
tr_q, tr_t = [], []
for i, r in enumerate(sg_pages):
    for q in r["qs"][:MAX_Q_PER_PAGE]:
        tr_q.append(q)
        tr_t.append(i)
VAL_T = list(range(len(val_pages)))

vdoc_tr_h = {r["h"] for r in sg_pages}
assert vdoc_tr_h.isdisjoint(val_h) and vdoc_tr_h.isdisjoint(test_h) and val_h.isdisjoint(test_h), "Rò rỉ hash V-Doc!"
assert {r["id"] for r in train_pages}.isdisjoint({r["id"] for r in test_pages}), "Rò rỉ id V-Doc!"

# Bộ kiểm tra tiếng Anh: ViDoRe DocVQA
EN_DOCS, EN_Q, EN_T = [], [], []
en_h = set()
if RUN_EN_CHECK:
    idx = {}
    os.makedirs(f"{DATA_DIR}/en", exist_ok=True)
    for row in load_dataset("vidore/docvqa_test_subsampled", split="test"):
        if row["image_filename"] not in idx:
            p = f"{DATA_DIR}/en/{len(EN_DOCS)}.png"
            row["image"].save(p)
            idx[row["image_filename"]] = len(EN_DOCS)
            EN_DOCS.append(p)
            en_h.add(img_hash(p))
        if row["query"]:
            EN_Q.append(row["query"])
            EN_T.append(idx[row["image_filename"]])

norm_q = lambda s: re.sub(r"\s+", " ", str(s).strip().lower())
forbidden_q_norm = {norm_q(q) for q in (TEST_Q + EN_Q)}
forbidden_hashes = test_h | val_h | en_h | vdoc_tr_h

print(f"V-Doc -> Train: {len(sg_pages)} trang ({len(tr_q)} câu) | Val: {len(val_pages)} | Test: {len(TEST_Q)} trang")
print(f"EN DocVQA -> {len(EN_Q)} câu / {len(EN_DOCS)} trang | Tập hash cấm leakage: {len(forbidden_hashes)}")
print("Khớp TEST_Q[0]:", TEST_Q[0])

## 2. Hàm đánh giá (giữ nguyên Giai đoạn 2) + thống kê chuẩn hoá
- Giữ nguyên `score_ragged_maxsim`, `ranks_from_scores` (xếp hạng bi quan), `per_query`, `metrics`.
- **Mới:** randomization test có cặp hai phía, `p = (b+1)/(B+1)` (không bao giờ in `0.000`), hiệu chỉnh **Holm** cho cả họ so sánh, CI95 bootstrap cho cả chênh lệch, điểm từng câu được trung bình qua seed trước khi kiểm định.
- Kết quả ghi vào file JSON **riêng của Giai đoạn 3**; file Giai đoạn 2 chỉ được đọc để đối chiếu.

In [ ]:
KEYS = ["nDCG@5", "R@1", "R@5", "MRR@10"]
MB_PER_1K = lambda vec, dim=128: vec * dim * 2 * 1000 / 2**20

def load_img(x):
    return (x if isinstance(x, Image.Image) else Image.open(x)).convert("RGB")

def to_dev(batch, model, cast_mask=False):
    p = next(model.parameters())
    dev, dt = p.device, p.dtype
    out = {}
    for k, v in batch.items():
        if k == "labels":
            continue
        if not torch.is_tensor(v):
            out[k] = v
            continue
        is_float = v.is_floating_point() or (cast_mask and k == "attention_mask")
        out[k] = v.to(dev, dtype=dt) if is_float else v.to(dev)
    return out

@torch.no_grad()
def encode(model, proc, items, is_image, bs=4, cast_mask=False):
    embs = []
    for i in tqdm(range(0, len(items), bs), desc="pages" if is_image else "queries", leave=False):
        chunk = items[i:i + bs]
        b = proc.process_images([load_img(x) for x in chunk]) if is_image else proc.process_queries(chunk)
        b = to_dev(b, model, cast_mask=cast_mask)
        e = model(**b)
        if isinstance(e, (tuple, list)):
            e = e[0]
        elif hasattr(e, "embeddings"):
            e = e.embeddings
        has_mask = ("attention_mask" in b) and torch.is_tensor(b["attention_mask"]) and (e.shape[1] == b["attention_mask"].shape[1])
        if has_mask:
            for x, m in zip(e, b["attention_mask"]):
                embs.append(x[m.bool()].to("cpu", torch.bfloat16))
        else:
            for x in e:
                valid = x.float().norm(dim=-1) > 1e-4
                embs.append((x[valid] if valid.any() else x).to("cpu", torch.bfloat16))
    return embs

def _pad_with_mask(seq_list):
    lens = [x.shape[0] for x in seq_list]
    padded = pad_sequence([x.float() for x in seq_list], batch_first=True)
    idx = torch.arange(padded.shape[1])[None, :]
    mask = idx < torch.tensor(lens)[:, None]
    return padded, mask

@torch.no_grad()
def score_ragged_maxsim(Q_list, D_list, G_list=None, q_chunk=32, d_chunk=128, device=DEVICE):
    """Tính MaxSim cho danh sách tensor độ dài động, có hỗ trợ trọng số cổng G_list."""
    use_cuda = str(device).startswith("cuda") and torch.cuda.is_available()
    comp_dt = torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported()) else (torch.float16 if use_cuda else torch.float32)
    out = torch.empty(len(Q_list), len(D_list), dtype=torch.float32)
    for i in range(0, len(Q_list), q_chunk):
        q_pad, q_mask = _pad_with_mask(Q_list[i:i + q_chunk])
        q_dev = q_pad.to(device, dtype=comp_dt)
        qm_dev = q_mask.to(device)
        if G_list is not None:
            g_pad = pad_sequence([g.float() for g in G_list[i:i + q_chunk]], batch_first=True).to(device)
            g_eff = g_pad * qm_dev.float()
        else:
            g_eff = qm_dev.float()
        for j in range(0, len(D_list), d_chunk):
            d_pad, d_mask = _pad_with_mask(D_list[j:j + d_chunk])
            d_dev = d_pad.to(device, dtype=comp_dt)
            dm_dev = d_mask.to(device)
            sim = torch.einsum("bqd,cpd->bcqp", q_dev, d_dev).float()
            sim = sim.masked_fill(~dm_dev[None, :, None, :], -1e9)
            max_sim = sim.amax(dim=-1) * g_eff[:, None, :]
            out[i:i + q_chunk, j:j + d_chunk] = max_sim.sum(dim=-1).cpu()
    return out

def ranks_from_scores(S, targets):
    S = torch.as_tensor(S, dtype=torch.float32)
    t = torch.as_tensor(targets)
    pos = S[torch.arange(len(t)), t]
    return (S >= pos[:, None]).sum(1).numpy()

def per_query(r, key):
    r = np.asarray(r, dtype=float)
    return {
        "nDCG@5": np.where(r <= 5, 1 / np.log2(r + 1), 0),
        "R@1": (r <= 1) * 1.0,
        "R@5": (r <= 5) * 1.0,
        "MRR@10": np.where(r <= 10, 1 / r, 0),
    }[key] * 100

def metrics(r):
    return {k: per_query(r, k).mean() for k in KEYS}

def boot(v, n=2000):
    return v[np.random.default_rng(0).integers(0, len(v), (n, len(v)))].mean(1)

def ci(r, key="nDCG@5"):
    return np.percentile(boot(per_query(r, key)), [2.5, 97.5])

def paired_p(ra, rb, key="nDCG@5"):
    b = boot(per_query(ra, key) - per_query(rb, key))
    return min(1.0, 2 * min((b <= 0).mean(), (b >= 0).mean()))

def vec_per_page(D):
    return float(np.mean([len(x) for x in D]))

def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def sync_ms(fn, n):
    use_cuda = torch.cuda.is_available()
    fn()
    if use_cuda:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = fn()
    if use_cuda:
        torch.cuda.synchronize()
    return out, (time.perf_counter() - t0) * 1000 / n

# ---- Unit tests nhanh cho score_ragged_maxsim (có và không có G_list) ----
_rng = torch.Generator().manual_seed(0)
_q_rag = [F.normalize(torch.randn(3, 16, generator=_rng), dim=-1), F.normalize(torch.randn(7, 16, generator=_rng), dim=-1)]
_d_rag = [F.normalize(torch.randn(5, 16, generator=_rng), dim=-1), F.normalize(torch.randn(11, 16, generator=_rng), dim=-1)]
_g_rag = [torch.tensor([0.5, 1.8, 0.7]), torch.linspace(0.6, 1.4, 7)]
_s_no_g = score_ragged_maxsim(_q_rag, _d_rag, G_list=None, q_chunk=1, d_chunk=1, device="cpu")
_s_no_g_ref = torch.tensor([[(q @ d.T).max(dim=-1).values.sum().item() for d in _d_rag] for q in _q_rag])
assert torch.allclose(_s_no_g, _s_no_g_ref, atol=1e-3)
_s_with_g = score_ragged_maxsim(_q_rag, _d_rag, G_list=_g_rag, q_chunk=1, d_chunk=1, device="cpu")
_s_with_g_ref = torch.tensor([[((q @ d.T).max(dim=-1).values * g).sum().item() for d in _d_rag] for q, g in zip(_q_rag, _g_rag)])
assert torch.allclose(_s_with_g, _s_with_g_ref, atol=1e-3)

# ---- Nạp RESULTS của notebook này (file JSON RIÊNG của Giai đoạn 3; không kế thừa và không ghi đè file Giai đoạn 1–2) ----
RESULTS = json.load(open(RES_PATH, encoding="utf-8")) if os.path.exists(RES_PATH) else {}
RESULTS_V2 = json.load(open(RES_V2_PATH, encoding="utf-8")) if os.path.exists(RES_V2_PATH) else {}
print(f"Kết quả Giai đoạn 2 (chỉ đọc, để đối chiếu): {len(RESULTS_V2)} mốc từ {RES_V2_PATH}")

def save_result(name, **kw):
    RESULTS[name] = {k: (None if v is None else np.asarray(v).tolist()) for k, v in kw.items()}
    with open(RES_PATH, "w", encoding="utf-8") as f:
        json.dump(RESULTS, f, ensure_ascii=False)

ocr_path = f"{SAVE_DIR}/ocr_test_{len(TEST_DOCS)}.json"
OCR_TEXTS = json.load(open(ocr_path, encoding="utf-8")) if os.path.exists(ocr_path) else None
print(f"Đã nạp {len(RESULTS)} mốc kết quả sẵn có:", list(RESULTS))

# =====================================================================================
# Thống kê chuẩn hoá cho IR (Giai đoạn 3)
#  - Điểm từng câu hỏi được TRUNG BÌNH QUA CÁC SEED trước, rồi mới kiểm định (không gộp seed×query thành mẫu độc lập)
#  - Paired randomization (sign-flip) test hai phía: p = (b+1)/(B+1) -> không bao giờ ra 0
#  - Holm–Bonferroni cho toàn bộ họ phép so sánh trong bảng
#  - CI95 bootstrap theo câu hỏi cho trung bình và cho CHÊNH LỆCH
# =====================================================================================
def pq_mean(v, key="nDCG@5"):
    """v: ranks 1 seed (list[int]) hoặc nhiều seed (list[list[int]]) -> điểm từng câu hỏi trung bình qua seed."""
    return np.mean([per_query(r, key) for r in np.atleast_2d(np.asarray(v, dtype=float))], 0)

def seed_scores(v, key="nDCG@5"):
    return [float(per_query(r, key).mean()) for r in np.atleast_2d(np.asarray(v, dtype=float))]

def paired_randomization_p(a, b, n_perm=None, seed=0):
    n_perm = N_PERM if n_perm is None else n_perm
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    n = len(d)
    obs = abs(d.mean())
    if n == 0 or obs < 1e-12:
        return 1.0
    rng = np.random.default_rng(seed)
    hits, done = 0, 0
    while done < n_perm:
        m = min(10000, n_perm - done)
        signs = rng.integers(0, 2, size=(m, n), dtype=np.int8).astype(np.float64) * 2.0 - 1.0
        hits += int((np.abs(signs @ d) / n >= obs - 1e-12).sum())
        done += m
    return (hits + 1) / (n_perm + 1)

def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    idx = np.where(ok)[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]
    m, running = len(order), 0.0
    for k, i in enumerate(order):
        running = max(running, (m - k) * p[i])
        out[i] = min(1.0, running)
    return out

def boot_ci(v, n=None, seed=0):
    n = N_BOOT if n is None else n
    v = np.asarray(v, dtype=float)
    bs = v[np.random.default_rng(seed).integers(0, len(v), (n, len(v)))].mean(1)
    return np.percentile(bs, [2.5, 97.5])

def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return None
    return f"{p:.1e}" if p < 1e-3 else f"{p:.4f}"

# ---- Tự kiểm tra nhanh (chạy trên CPU, vài giây) ----
_a = np.r_[np.ones(200), np.zeros(100)] * 100
_b = np.r_[np.ones(100), np.zeros(200)] * 100
assert paired_randomization_p(_a, _b, n_perm=2000) <= 1 / 2001 + 1e-9
assert paired_randomization_p(_a, _a, n_perm=2000) == 1.0
_c = np.tile([100.0, 0.0, 100.0, 0.0, 0.0, 100.0], 30)
assert 0.05 < paired_randomization_p(_c, np.roll(_c, 1), n_perm=2000) <= 1.0  # chênh lệch nhỏ -> không có ý nghĩa
_h = holm_adjust([0.01, 0.04, np.nan, 0.03])
assert np.allclose(_h[[0, 1, 3]], [0.03, 0.06, 0.06]) and np.isnan(_h[2])
print("✓ Tự kiểm tra thống kê (randomization test + Holm) OK")

## 3. Kiến trúc & vòng train Giai đoạn 2 (chép nguyên văn, không sửa)
Cần cho các dòng đối chứng: nhánh query của v2 (baseline) và kiến trúc v2 chạy lại với `'Query: '`.

In [ ]:
def score_masked_maxsim(q_emb, q_mask, d_emb, d_mask, gates=None, q_chunk_size=8, d_chunk_size=64):
    B = q_emb.shape[0]
    C = d_emb.shape[0]
    q_fl, d_fl = q_emb.float(), d_emb.float()
    g_eff = (gates.float() * q_mask.float()) if gates is not None else q_mask.float()
    score_rows = []
    for i in range(0, B, q_chunk_size):
        q_sub = q_fl[i:i + q_chunk_size]
        g_sub = g_eff[i:i + q_chunk_size].unsqueeze(1)
        doc_cols = []
        for j in range(0, C, d_chunk_size):
            d_sub = d_fl[j:j + d_chunk_size]
            dm_sub = d_mask[j:j + d_chunk_size]
            sim = torch.einsum("bqd,cpd->bcqp", q_sub, d_sub)
            sim = sim.masked_fill(~dm_sub[None, :, None, :], -1e9)
            max_sim = sim.max(dim=-1).values
            doc_cols.append((max_sim * g_sub).sum(dim=-1))
        score_rows.append(torch.cat(doc_cols, dim=1))
    return torch.cat(score_rows, dim=0)

class DynamicMultiVectorAdapter(nn.Module):
    def __init__(self, dim=128, rank=32, window_size=8, dropout=0.1,
                 doc_mode="dyn_merge", use_doc_adapter=True,
                 use_subword_attn=True, use_query_gate=True):
        super().__init__()
        assert doc_mode in ("none", "dyn_merge")
        assert window_size == 8, "2 vòng gộp đôi 8 -> 4 -> 2 yêu cầu window_size=8"
        self.dim = dim
        self.window_size = window_size
        self.doc_mode = doc_mode
        self.use_doc_adapter = use_doc_adapter
        self.use_subword_attn = use_subword_attn
        self.use_query_gate = use_query_gate

        # Nhánh 1A: Masked 1D Context Conv & Position-Free Saliency
        self.doc_conv = nn.Sequential(
            nn.Conv1d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False),
            nn.GroupNorm(8, dim),
            nn.GELU(),
            nn.Conv1d(dim, rank, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(rank, dim, kernel_size=1, bias=False),
        )
        self.alpha_conv = nn.Parameter(torch.tensor(0.02))

        self.saliency_head = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(16, 1, kernel_size=1),
        )
        nn.init.normal_(self.saliency_head[-1].weight, std=0.02)
        nn.init.zeros_(self.saliency_head[-1].bias)
        self.saliency_temp = nn.Parameter(torch.tensor(0.30))

        # Nhánh 1B: Window Cross-Attention Refiner
        self.merge_norm = nn.LayerNorm(dim)
        self.merge_q = nn.Linear(dim, rank, bias=False)
        self.merge_k = nn.Linear(dim, rank, bias=False)
        self.merge_v = nn.Linear(dim, rank, bias=False)
        self.merge_out = nn.Linear(rank, dim, bias=False)
        self.alpha_merge = nn.Parameter(torch.tensor(0.04))

        # Nhánh 2A: Low-Rank Bidirectional Subword Attention
        self.sub_norm = nn.LayerNorm(dim)
        self.sub_q = nn.Linear(dim, rank, bias=False)
        self.sub_k = nn.Linear(dim, rank, bias=False)
        self.sub_v = nn.Linear(dim, rank, bias=False)
        self.sub_out = nn.Linear(rank, dim, bias=False)
        self.alpha_sub = nn.Parameter(torch.tensor(0.04))

        # Nhánh 2B: Position-Free Query Context Gate (5 đặc trưng, ĐÃ BỎ f6_pos)
        self.gate_mlp = nn.Sequential(
            nn.Linear(dim + 5, 48),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(48, 1),
        )
        nn.init.normal_(self.gate_mlp[-1].weight, std=0.01)
        nn.init.zeros_(self.gate_mlp[-1].bias)
        self.gate_scale = nn.Parameter(torch.tensor(0.30))

    def _merge_windows(self, win_x, win_w, win_m):
        """Gộp 2 vòng trong từng cửa sổ độ dài 8: 8 -> 4 -> 2, không để padding lọt vào."""
        B, S, _, D = win_x.shape
        x = win_x
        w = win_w * win_m.float()
        m = win_m
        for _ in range(2):
            A, wA, mA = x[:, :, 0::2, :], w[:, :, 0::2], m[:, :, 0::2]
            B_set, wB, mB = x[:, :, 1::2, :], w[:, :, 1::2], m[:, :, 1::2]
            sim = torch.einsum("bsid,bsjd->bsij", B_set, A)
            has_valid_A = mA.any(dim=-1, keepdim=True)
            sim = sim.masked_fill((~mA.unsqueeze(2)) & has_valid_A.unsqueeze(2), -1e9)
            best_a = sim.argmax(dim=-1)

            merged_A = A * wA.unsqueeze(-1)
            merged_A.scatter_add_(2, best_a.unsqueeze(-1).expand(-1, -1, -1, D), B_set * wB.unsqueeze(-1))

            new_w = wA.clone()
            new_w.scatter_add_(2, best_a, wB)
            new_m = new_w > 1e-6
            x = F.normalize(merged_A, p=2, dim=-1) * new_m.float().unsqueeze(-1)
            w = new_w
            m = new_m
        return x, m

    def forward_docs(self, E_d, d_mask):
        d_mask_b = d_mask.bool()
        d_mask_fl = d_mask_b.float()
        x = F.normalize(E_d.float(), p=2, dim=-1) * d_mask_fl.unsqueeze(-1)

        if self.use_doc_adapter:
            x_b = x.permute(0, 2, 1)
            m_b = d_mask_fl.unsqueeze(1)
            num_d = d_mask_fl.sum(dim=1, keepdim=True).clamp(min=1.0)

            sum_local = F.avg_pool1d(x_b, kernel_size=3, stride=1, padding=1) * 3.0
            cnt_local = F.avg_pool1d(m_b, kernel_size=3, stride=1, padding=1) * 3.0
            mean_local = sum_local / cnt_local.clamp(min=1.0)
            diff_local3 = (x_b - mean_local).norm(dim=1, keepdim=True) * m_b

            doc_mean = x.sum(dim=1, keepdim=True) / num_d.unsqueeze(-1)
            diff_glob = (x - doc_mean).norm(dim=-1).unsqueeze(1) * m_b
            max_ch = x_b.abs().max(dim=1, keepdim=True).values * m_b

            sal_raw = torch.cat([diff_local3, diff_glob, max_ch], dim=1)
            sal_mean = (sal_raw * m_b).sum(dim=2, keepdim=True) / num_d.unsqueeze(-1)
            sal_var = (((sal_raw - sal_mean) ** 2) * m_b).sum(dim=2, keepdim=True) / num_d.unsqueeze(-1)
            sal_in = ((sal_raw - sal_mean) / (sal_var.sqrt() + 1e-5)) * m_b
            sal_logits = self.saliency_head(sal_in).squeeze(1) * d_mask_fl
            patch_weights = torch.exp(torch.clamp(self.saliency_temp, 0.05, 0.8) * torch.tanh(sal_logits)) * d_mask_fl

            conv_delta = self.doc_conv(x_b).permute(0, 2, 1) * d_mask_fl.unsqueeze(-1)
            x = F.normalize(x + 0.15 * torch.tanh(self.alpha_conv) * conv_delta, p=2, dim=-1) * d_mask_fl.unsqueeze(-1)
        else:
            sal_logits = None
            patch_weights = d_mask_fl

        if self.doc_mode == "none":
            return x, d_mask_b

        B, L, D = x.shape
        W = self.window_size
        pad_len = (W - (L % W)) % W
        if pad_len > 0:
            x = F.pad(x, (0, 0, 0, pad_len))
            patch_weights = F.pad(patch_weights, (0, pad_len))
            d_mask_b = F.pad(d_mask_b, (0, pad_len), value=False)
            if sal_logits is not None:
                sal_logits = F.pad(sal_logits, (0, pad_len))

        S = (L + pad_len) // W
        win_x = x.view(B, S, W, D)
        win_w = patch_weights.view(B, S, W)
        win_m = d_mask_b.view(B, S, W)

        seeds, seed_mask = self._merge_windows(win_x, win_w, win_m)

        if self.use_doc_adapter:
            win_sal = sal_logits.view(B, S, W)
            sn = self.merge_norm(win_x)
            qn = self.merge_norm(seeds)
            Q, K, V = self.merge_q(qn), self.merge_k(sn), self.merge_v(sn)
            attn_logits = torch.einsum("bsqr,bskr->bsqk", Q, K) / (Q.shape[-1] ** 0.5) + 0.3 * torch.tanh(win_sal).unsqueeze(2)
            attn_logits = attn_logits.masked_fill(~win_m.unsqueeze(2), -1e4)
            attn_weights = F.softmax(attn_logits, dim=-1) * win_m.float().unsqueeze(2)
            attn_out = self.merge_out(torch.einsum("bsqk,bskr->bsqr", attn_weights, V))
            merged = F.normalize(seeds + 0.25 * torch.tanh(self.alpha_merge) * attn_out, p=2, dim=-1) * seed_mask.float().unsqueeze(-1)
        else:
            merged = seeds

        return merged.reshape(B, S * 2, D), seed_mask.reshape(B, S * 2)

    def forward_queries(self, E_q, q_mask):
        q_mask_b = q_mask.bool()
        valid_float = q_mask_b.float()
        q_float = F.normalize(E_q.float(), p=2, dim=-1) * valid_float.unsqueeze(-1)

        if self.use_subword_attn:
            qn = self.sub_norm(q_float)
            Q, K, V = self.sub_q(qn), self.sub_k(qn), self.sub_v(qn)
            scores = torch.einsum("bqr,bkr->bqk", Q, K) / (Q.shape[-1] ** 0.5)
            scores = scores.masked_fill(~q_mask_b.unsqueeze(1), -1e4)
            attn_w = F.softmax(scores, dim=-1) * valid_float.unsqueeze(-1)
            sub_delta = self.sub_out(torch.einsum("bqk,bkr->bqr", attn_w, V))
            q_contextual = F.normalize(q_float + 0.20 * torch.tanh(self.alpha_sub) * sub_delta, p=2, dim=-1) * valid_float.unsqueeze(-1)
        else:
            q_contextual = q_float

        if self.use_query_gate:
            B, L, _ = q_contextual.shape
            num_valid = valid_float.sum(dim=-1, keepdim=True).clamp(min=1.0)
            q_global = F.normalize(q_contextual.sum(dim=1, keepdim=True) / num_valid.unsqueeze(-1), p=2, dim=-1)

            f1_dist = 1.0 - (q_contextual * q_global).sum(dim=-1, keepdim=True)
            prev_q = torch.roll(q_contextual, shifts=1, dims=1); prev_q[:, 0, :] = 0.0
            next_q = torch.roll(q_contextual, shifts=-1, dims=1); next_q[:, -1, :] = 0.0
            f2_prev = (q_contextual * prev_q).sum(dim=-1, keepdim=True)
            f3_next = (q_contextual * next_q).sum(dim=-1, keepdim=True)

            sim_mat = torch.einsum("bqd,bkd->bqk", q_contextual, q_contextual)
            eye_mask = torch.eye(L, device=E_q.device).bool().unsqueeze(0)
            peer_mask = q_mask_b.unsqueeze(1) & (~eye_mask)
            f4_mean = sim_mat.masked_fill(~peer_mask, 0.0).sum(dim=-1, keepdim=True) / (num_valid.unsqueeze(-1) - 1.0).clamp(min=1.0)
            f5_max = sim_mat.masked_fill(~peer_mask, -1.0).max(dim=-1, keepdim=True).values.clamp(min=0.0)

            gate_in = torch.cat([q_contextual, f1_dist, f2_prev, f3_next, f4_mean, f5_max], dim=-1) * valid_float.unsqueeze(-1)
            gate_delta = torch.tanh(self.gate_mlp(gate_in)).squeeze(-1) * valid_float
            scale = torch.clamp(self.gate_scale, 0.05, 0.50)
            raw_gates = (1.0 + scale * gate_delta) * valid_float
            gates_norm = raw_gates * (num_valid / (raw_gates.sum(dim=-1, keepdim=True) + 1e-6))
            return q_contextual, gates_norm, gate_delta, q_float
        else:
            return q_contextual, None, torch.zeros(1, device=E_q.device), q_float

    def forward(self, E_q, q_mask, E_d, d_mask):
        if self.use_doc_adapter:
            d_emb, out_d_mask = self.forward_docs(E_d, d_mask)
        else:
            with torch.no_grad():
                d_emb, out_d_mask = self.forward_docs(E_d, d_mask)

        q_cs = 4 if self.doc_mode == "none" else SCORE_CHUNK
        if self.use_subword_attn or self.use_query_gate:
            q_emb, gates_norm, gate_delta, q_orig = self.forward_queries(E_q, q_mask)
            scores = score_masked_maxsim(q_emb, q_mask, d_emb, out_d_mask, gates=gates_norm,
                                         q_chunk_size=q_cs, d_chunk_size=DOC_CHUNK)
            return scores, gate_delta, q_emb, q_orig
        else:
            q_orig = F.normalize(E_q.float(), p=2, dim=-1) * q_mask.float().unsqueeze(-1)
            scores = score_masked_maxsim(q_orig, q_mask, d_emb, out_d_mask, gates=None,
                                         q_chunk_size=q_cs, d_chunk_size=DOC_CHUNK)
            return scores, torch.zeros(1, device=E_q.device), q_orig, q_orig

_tmp_net = DynamicMultiVectorAdapter(dim=128, rank=32)
print(f"Tổng tham số DynamicMultiVectorAdapter: {sum(p.numel() for p in _tmp_net.parameters()):,} (~{sum(p.numel() for p in _tmp_net.parameters()) * 4 / 1024:.1f} KB)")
del _tmp_net

In [ ]:
@torch.no_grad()
def eval_adapter_ragged(net, Q_list, D_list, targets, doc_bs=32, q_bs=64):
    net.eval()
    D_comp = []
    for i in range(0, len(D_list), doc_bs):
        d_pad, d_mask = _pad_with_mask(D_list[i:i + doc_bs])
        d_out, m_out = net.forward_docs(d_pad.to(DEVICE), d_mask.to(DEVICE))
        d_out_cpu = d_out.to("cpu", torch.bfloat16)
        m_out_cpu = m_out.cpu()
        for x, m in zip(d_out_cpu, m_out_cpu):
            D_comp.append(x[m])

    Q_adapt, G_adapt = [], []
    use_g = net.use_query_gate
    for i in range(0, len(Q_list), q_bs):
        q_pad, q_mask = _pad_with_mask(Q_list[i:i + q_bs])
        q_out, g_out, _, _ = net.forward_queries(q_pad.to(DEVICE), q_mask.to(DEVICE))
        q_out_cpu = q_out.to("cpu", torch.bfloat16)
        q_mask_cpu = q_mask.cpu()
        g_cpu = g_out.float().cpu() if (use_g and g_out is not None) else None
        for idx_q in range(len(q_out_cpu)):
            m = q_mask_cpu[idx_q]
            Q_adapt.append(q_out_cpu[idx_q][m])
            if g_cpu is not None:
                G_adapt.append(g_cpu[idx_q][m])

    S = score_ragged_maxsim(Q_adapt, D_comp, G_list=G_adapt if use_g else None)
    return ranks_from_scores(S, targets), D_comp

@torch.no_grad()
def time_doc_compression(net, D_list, doc_bs=32):
    net.eval()
    batches = []
    for i in range(0, len(D_list), doc_bs):
        d_pad, d_mask = _pad_with_mask(D_list[i:i + doc_bs])
        batches.append((d_pad.to(DEVICE), d_mask.to(DEVICE)))
    _, ms = sync_ms(lambda: [net.forward_docs(xb, mb) for xb, mb in batches], len(D_list))
    del batches
    return ms

def mine_hard_negatives(Q_tr_list, D_tr_list, tr_t_np, tr_h_ids, k=HARD_NEG):
    S_tea = score_ragged_maxsim(Q_tr_list, D_tr_list)
    print("ColQwen2.5 gốc trên tập train:",
          {m: round(float(v), 1) for m, v in metrics(ranks_from_scores(S_tea, tr_t_np)).items()})
    same = tr_h_ids[None, :] == tr_h_ids[tr_t_np][:, None]
    hard = S_tea.masked_fill(torch.as_tensor(same), -1e9).topk(k, dim=1).indices.numpy()
    del S_tea, same
    return hard

def train_qwen_adapter(Q_tr_list, D_tr_list, tr_t_np, tr_h_ids, hard_neg_np,
                       Q_val_list, D_val_list, doc_mode, use_doc_adapter,
                       use_subword_attn, use_query_gate, use_kd, seed):
    torch.manual_seed(seed)
    gen = np.random.default_rng(seed)
    emb_dim = int(D_tr_list[0].shape[-1])
    net = DynamicMultiVectorAdapter(
        dim=emb_dim, rank=32, window_size=8, dropout=0.1,
        doc_mode=doc_mode, use_doc_adapter=use_doc_adapter,
        use_subword_attn=use_subword_attn, use_query_gate=use_query_gate,
    ).to(DEVICE)

    Nq = len(Q_tr_list)
    steps_per_epoch = int(np.ceil(Nq / BQ))
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = steps_per_epoch * WARMUP_EPOCHS

    opt = torch.optim.AdamW(net.parameters(), lr=LR_ADAPTER, weight_decay=WEIGHT_DECAY)
    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    best, best_state = -1.0, None

    for ep in range(EPOCHS):
        net.train()
        perm = gen.permutation(Nq)
        ep_loss, n_steps = 0.0, 0
        for i in range(0, Nq, BQ):
            idx = perm[i:i + BQ]
            pos = tr_t_np[idx]
            cand = np.unique(np.concatenate([pos, hard_neg_np[idx].reshape(-1)]))
            tgt = torch.as_tensor(np.searchsorted(cand, pos), dtype=torch.long, device=DEVICE)

            pos_h = tr_h_ids[pos][:, None]
            cand_h = tr_h_ids[cand][None, :]
            fn = torch.as_tensor((cand_h == pos_h) & (cand[None, :] != pos[:, None]), device=DEVICE)

            q_pad, q_mask = _pad_with_mask([Q_tr_list[j] for j in idx])
            d_pad, d_mask = _pad_with_mask([D_tr_list[j] for j in cand])
            q_pad, q_mask = q_pad.to(DEVICE), q_mask.to(DEVICE)
            d_pad, d_mask = d_pad.to(DEVICE), d_mask.to(DEVICE)

            s, g_delta, q_emb, q_orig = net(q_pad, q_mask, d_pad, d_mask)
            s = s.masked_fill(fn, -1e4)
            loss = F.cross_entropy(s / TEMP_CE, tgt)

            if use_kd:
                with torch.no_grad():
                    d_tea = F.normalize(d_pad.float(), p=2, dim=-1) * d_mask.float().unsqueeze(-1)
                    t = score_masked_maxsim(
                        q_orig, q_mask, d_tea, d_mask, gates=None, q_chunk_size=SCORE_CHUNK, d_chunk_size=DOC_CHUNK
                    ).masked_fill(fn, -1e4)
                kd = F.kl_div(F.log_softmax(s / TEMP_KD, -1), F.softmax(t / TEMP_KD, -1), reduction="batchmean") * (TEMP_KD ** 2)
                anchor = (1.0 - (q_emb * q_orig).sum(dim=-1))[q_mask].mean()
                loss = 0.50 * loss + 0.50 * kd + ANCHOR_W * anchor

            if use_query_gate:
                loss = loss + 0.005 * g_delta.abs().mean()

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            sched.step()
            ep_loss += float(loss.item())
            n_steps += 1

        val_ranks, _ = eval_adapter_ragged(net, Q_val_list, D_val_list, VAL_T)
        nd = float(metrics(val_ranks)["nDCG@5"])
        if nd > best:
            best, best_state = nd, copy.deepcopy(net.state_dict())
        print(f"  epoch {ep + 1}/{EPOCHS}  loss {ep_loss / max(1, n_steps):.4f}  val nDCG@5 {nd:.1f}")

    net.load_state_dict(best_state)
    return net.eval()

## 4. Cache embedding + **sửa giao thức query** (`'Query: '`) + bộ test ngoài miền
- `colpali-engine` ≥ 0.3.13 đặt `query_prefix = ""`, trong khi `vidore/colqwen2.5-v0.2` được train với `"Query: " + câu hỏi + 10×<|endoftext|>` → Giai đoạn 2 đo backbone **dưới giao thức sai**. Ở đây chỉ encode lại **câu hỏi** (embedding trang giữ nguyên cache Giai đoạn 2).
- Lưu thêm `content mask` cho từng câu (token nào thuộc câu hỏi, token nào là tiền tố/augmentation) để phân tích gate đúng.
- **OOD:** `ViInfographicVQA` single-image test (193 infographic thật / 1.374 câu hỏi do người kiểm duyệt) — khác miền sách giáo khoa và khác nguồn sinh câu hỏi của V-Doc.

In [ ]:
def load_model_safe(loader_fn, proc, sample_imgs, sample_qs, cast_mask=False):
    if torch.cuda.is_available():
        if PREFER_FP16_ON_T4 and not torch.cuda.is_bf16_supported():
            dtypes = [torch.float16, torch.bfloat16]
        else:
            dtypes = [torch.bfloat16]
    else:
        dtypes = [torch.float32]

    last_err = None
    for dt in dtypes:
        m = None
        try:
            print(f"  Thử nạp model với dtype={dt}...")
            m = loader_fn(dt).eval()
            out_d = encode(m, proc, sample_imgs[:2], True, bs=2, cast_mask=cast_mask)
            out_q = encode(m, proc, sample_qs[:2], False, bs=2, cast_mask=cast_mask)
            if all(torch.isfinite(x).all() and x.float().norm(dim=-1).mean() > 1e-3 for x in out_d + out_q):
                print(f"  ✓ Sanity check thành công với dtype={dt} | shape trang mẫu: {tuple(out_d[0].shape)}")
                return m, dt
            raise RuntimeError(f"Sanity check phát hiện NaN/Inf hoặc vector 0 với dtype={dt}")
        except Exception as exc:
            last_err = exc
            print(f"  ✗ Lỗi với dtype={dt}: {exc}")
            if m is not None:
                del m
            free()
    raise RuntimeError(f"Không nạp được model. Lỗi cuối: {last_err}")
# =====================================================================================
# Cache embedding ColQwen2.5 (tái sử dụng cache Giai đoạn 2) + encode lại QUERY có tiền tố "Query: " + bộ OOD
# =====================================================================================
from colpali_engine.models import ColQwen2_5, ColQwen2_5_Processor

PROC_Q = ColQwen2_5_Processor.from_pretrained(QWEN_ID)
_LIB_PREFIX = getattr(PROC_Q, "query_prefix", "") or ""
print(f"query_prefix mặc định của colpali-engine đang cài: {_LIB_PREFIX!r}")

def make_prefixed_processor():
    """Processor mới mà process_queries() tự thêm 'Query: ' (nếu thư viện chưa thêm) -> khớp giao thức train của v0.2."""
    p = ColQwen2_5_Processor.from_pretrained(QWEN_ID)
    if "query" not in (getattr(p, "query_prefix", "") or "").lower():
        p.query_prefix = QUERY_PREFIX
    return p

PROC_QP = make_prefixed_processor()
_demo = PROC_QP.process_queries(["xin chào"])
_demo_txt = PROC_QP.tokenizer.decode(_demo["input_ids"][0])
print("Ví dụ chuỗi query thực sự đưa vào model:", repr(_demo_txt))
assert _demo_txt.count("Query") == 1, "Tiền tố 'Query:' bị thiếu hoặc bị lặp!"

def query_token_info(proc, texts):
    """Trả về (ids, content_mask) cho từng query, KHỚP 1-1 với vector embedding (cùng tokenizer/process_texts).
    content_mask = True cho token thuộc phần câu hỏi (không phải tiền tố 'Query:' và không phải 10 token augmentation)."""
    prefix = getattr(proc, "query_prefix", "") or ""
    suffix = proc.query_augmentation_token * 10
    ids_all, cm_all = [], []
    for t in texts:
        full = prefix + t + suffix
        enc = proc.tokenizer(full, return_offsets_mapping=True)
        lo, hi = len(prefix), len(prefix) + len(t)
        cm = [bool(e > lo and s < hi and e > s) for (s, e) in enc["offset_mapping"]]
        ids_all.append(list(enc["input_ids"]))
        cm_all.append(np.asarray(cm, dtype=bool))
    return ids_all, cm_all

def _check_align(embs, ids, name):
    bad = sum(int(len(e) != len(i)) for e, i in zip(embs, ids))
    if bad:
        print(f"  ⚠ {name}: {bad}/{len(embs)} query lệch độ dài token/embedding -> dùng content mask dự phòng")
    return bad

def _fallback_cm(e):
    cm = np.ones(len(e), dtype=bool)
    cm[-10:] = False
    cm[:2] = False
    return cm

def build_query_pack(model, proc, texts, bs=32, name="q"):
    Q = encode(model, proc, texts, False, bs=bs)
    ids, cms = query_token_info(proc, texts)
    if _check_align(Q, ids, name):
        cms = [cm if len(cm) == len(e) else _fallback_cm(e) for e, cm in zip(Q, cms)]
    return Q, ids, cms

_QWEN_MODEL = None
def get_qwen_model():
    """Nạp ColQwen2.5 zero-shot đúng 1 lần (chỉ khi thiếu cache)."""
    global _QWEN_MODEL
    if _QWEN_MODEL is None:
        _QWEN_MODEL, _ = load_model_safe(
            lambda dt: ColQwen2_5.from_pretrained(QWEN_ID, torch_dtype=dt, device_map=DEVICE),
            PROC_Q, TEST_DOCS[:2], TEST_Q[:2], cast_mask=False,
        )
    return _QWEN_MODEL

def release_qwen_model():
    global _QWEN_MODEL
    if _QWEN_MODEL is not None:
        del _QWEN_MODEL
        _QWEN_MODEL = None
        free()

# ---- 1. Cache Giai đoạn 2 (page + query KHÔNG tiền tố). Nếu thiếu thì encode y hệt Giai đoạn 2 ----
if not os.path.exists(QWEN_TEST_CACHE):
    print(">>> Thiếu cache test Giai đoạn 2 -> encode lại (giống hệt Giai đoạn 2)")
    m = get_qwen_model()
    t0 = time.perf_counter()
    _Dt = encode(m, PROC_Q, TEST_DOCS, True, bs=QWEN_IMG_BS)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    _enc_s = (time.perf_counter() - t0) / len(TEST_DOCS)
    torch.save({
        "D_test": _Dt, "Q_test": encode(m, PROC_Q, TEST_Q, False, bs=16),
        "D_en": encode(m, PROC_Q, EN_DOCS, True, bs=QWEN_IMG_BS), "Q_en": encode(m, PROC_Q, EN_Q, False, bs=16),
        "encode_s_per_page": _enc_s, "embed_dim": int(_Dt[0].shape[-1]), "dtype": "torch.bfloat16",
    }, QWEN_TEST_CACHE)
    del _Dt
if not os.path.exists(QWEN_TRVAL_CACHE):
    print(">>> Thiếu cache train/val Giai đoạn 2 -> encode lại (giống hệt Giai đoạn 2)")
    m = get_qwen_model()
    torch.save({
        "D_tr": encode(m, PROC_Q, [r["path"] for r in sg_pages], True, bs=QWEN_IMG_BS),
        "Q_tr": encode(m, PROC_Q, tr_q, False, bs=32),
        "D_val": encode(m, PROC_Q, [r["path"] for r in val_pages], True, bs=QWEN_IMG_BS),
        "Q_val": encode(m, PROC_Q, [r["qs"][0] for r in val_pages], False, bs=32),
    }, QWEN_TRVAL_CACHE)

pack_test = torch.load(QWEN_TEST_CACHE, map_location="cpu")
D_test, Q_test_np = pack_test["D_test"], pack_test["Q_test"]
D_en, Q_en_np = pack_test.get("D_en"), pack_test.get("Q_en")
ENC_S_PER_PAGE = pack_test.get("encode_s_per_page")
pack_trval = torch.load(QWEN_TRVAL_CACHE, map_location="cpu")
D_tr, Q_tr_np = pack_trval["D_tr"], pack_trval["Q_tr"]
D_val, Q_val_np = pack_trval["D_val"], pack_trval["Q_val"]
EMB_DIM = int(D_test[0].shape[-1])

# ---- 2. Query có tiền tố "Query: " (chỉ encode query, embedding trang giữ nguyên) ----
if not os.path.exists(QPREFIX_CACHE):
    print(">>> Encode lại toàn bộ query với tiền tố 'Query: ' ...")
    m = get_qwen_model()
    _qp = {}
    for key, texts in [("test", TEST_Q), ("val", [r["qs"][0] for r in val_pages]), ("tr", tr_q), ("en", EN_Q)]:
        Q, ids, cms = build_query_pack(m, PROC_QP, texts, bs=32, name=key)
        _qp[f"Q_{key}"], _qp[f"ids_{key}"], _qp[f"cm_{key}"] = Q, ids, cms
    torch.save(_qp, QPREFIX_CACHE)
qp = torch.load(QPREFIX_CACHE, map_location="cpu", weights_only=False)
Q_test, Q_val, Q_tr, Q_en = qp["Q_test"], qp["Q_val"], qp["Q_tr"], qp["Q_en"]
C_test, C_val, C_tr, C_en = qp["cm_test"], qp["cm_val"], qp["cm_tr"], qp["cm_en"]
IDS_test = qp["ids_test"]
assert len(Q_test) == len(TEST_Q) and len(Q_tr) == len(tr_q) and len(Q_en) == len(EN_Q)

# ---- 3. Bộ test ngoài miền ViInfographicVQA (single-image test) ----
def load_ood_viinfographic():
    ood_dir = f"{DATA_DIR}/viinfographic"
    js = hf_hub_download(OOD_REPO, "data/single_test.json", repo_type="dataset", local_dir=ood_dir)
    rows = json.load(open(js, encoding="utf-8"))
    rows = [r for r in rows if isinstance(r.get("question"), str) and len(r["question"].strip()) > 10 and r.get("image_path")]
    paths = sorted({r["image_path"] for r in rows})
    local = {}
    for ip in tqdm(paths, desc="tải ảnh ViInfographicVQA"):
        local[ip] = hf_hub_download(OOD_REPO, ip, repo_type="dataset", local_dir=ood_dir)
    pos = {ip: i for i, ip in enumerate(paths)}
    random.Random(SEED).shuffle(rows)
    rows = rows[:OOD_MAX_Q]
    return [local[p] for p in paths], [r["question"].strip() for r in rows], [pos[r["image_path"]] for r in rows]

D_ood = Q_ood = C_ood = None
OOD_DOCS, OOD_Q, OOD_T = [], [], []
if RUN_OOD:
    try:
        OOD_DOCS, OOD_Q, OOD_T = load_ood_viinfographic()
        ood_h = {img_hash(p) for p in OOD_DOCS}
        print(f"OOD ViInfographicVQA: {len(OOD_DOCS)} ảnh / {len(OOD_Q)} câu hỏi | "
              f"trùng hash với V-Doc train/val/test: {len(ood_h & (vdoc_tr_h | val_h | test_h))}")
        if not os.path.exists(OOD_CACHE):
            m = get_qwen_model()
            D_o = encode(m, PROC_Q, OOD_DOCS, True, bs=QWEN_IMG_BS)
            Q_o, ids_o, cm_o = build_query_pack(m, PROC_QP, OOD_Q, bs=32, name="ood")
            torch.save({"D_ood": D_o, "Q_ood": Q_o, "ids_ood": ids_o, "cm_ood": cm_o, "T_ood": OOD_T}, OOD_CACHE)
        _po = torch.load(OOD_CACHE, map_location="cpu", weights_only=False)
        D_ood, Q_ood, C_ood = _po["D_ood"], _po["Q_ood"], _po["cm_ood"]
        assert list(_po["T_ood"]) == list(OOD_T), "Thứ tự OOD không khớp cache -> xoá file OOD_CACHE rồi chạy lại"
    except Exception as exc:
        print(f"⚠ Bỏ qua bộ OOD do lỗi: {exc}")
        traceback.print_exc()
        D_ood = Q_ood = C_ood = None

release_qwen_model()
print(f"Sẵn sàng -> D_tr {len(D_tr)} | D_val {len(D_val)} | D_test {len(D_test)} | D_en {len(D_en)} | "
      f"D_ood {len(D_ood) if D_ood else 0} | dim {EMB_DIM} | {vec_per_page(D_test):.1f} vec/trang")

In [ ]:
# =====================================================================================
# Hierarchical Token Pooling ×4 (Clavié et al., 2024 — HierarchicalTokenPooler chính thức của colpali-engine)
# Cache riêng theo từng tập; tập train/val chỉ cần cho baseline "adapter + Hier×4".
# =====================================================================================
from colpali_engine.compression.token_pooling import HierarchicalTokenPooler
HIER_POOLER = HierarchicalTokenPooler()

def hier_pool(D_list, pool_factor=4):
    out = HIER_POOLER.pool_embeddings([x.float() for x in D_list], pool_factor=pool_factor, num_workers=os.cpu_count())
    return [x.to(torch.bfloat16) for x in out]

HIER = torch.load(HIER_CACHE, map_location="cpu") if os.path.exists(HIER_CACHE) else {}
_need = {"test": D_test, "en": D_en}
if D_ood is not None:
    _need["ood"] = D_ood
if NEED_HIER_TRAIN:
    _need.update({"tr": D_tr, "val": D_val})
_changed = False
for key, D in _need.items():
    if key not in HIER or len(HIER[key]) != len(D):
        t0 = time.perf_counter()
        HIER[key] = hier_pool(D, 4)
        HIER[f"ms_{key}"] = (time.perf_counter() - t0) * 1000 / len(D)
        _changed = True
        print(f"  Hier×4 [{key}]: {len(D)} trang | {vec_per_page(HIER[key]):.1f} vec/trang | {HIER[f'ms_{key}']:.1f} ms/trang (CPU x{os.cpu_count()})")
if _changed:
    torch.save(HIER, HIER_CACHE)
print("Hier×4 sẵn sàng cho:", [k for k in HIER if not k.startswith("ms_")])

## 5. Mốc zero-shot chuẩn hoá & Hier×4
Báo cáo cả hai giao thức (có / không `'Query: '`) để thấy giao thức đổi mốc bao nhiêu. **Mọi Δ và p trong bảng cuối đều so với mốc CÓ tiền tố.**

In [ ]:
# =====================================================================================
# Mốc zero-shot chuẩn hoá: có / không tiền tố "Query: " (kiểm chứng giao thức) + Hier×4
# =====================================================================================
REF_ZS = "ZS: ColQwen2.5-3B (+'Query:')"
REF_ZS_NOPFX = "ZS: ColQwen2.5-3B (không tiền tố — giao thức v2)"
REF_HIER = "ZS: ColQwen2.5 + Hier×4 (+'Query:')"
DEVICE_NAME = torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU"

def rank_lists(Q, D, T, G=None):
    return ranks_from_scores(score_ragged_maxsim(Q, D, G_list=G), T)

def eval_triplet(Qs, Ds, G=None):
    """Qs/Ds: dict {'vi','en','ood'} -> list; trả về ranks cho từng bộ có sẵn."""
    out = {"vi": rank_lists(Qs["vi"], Ds["vi"], TEST_T)}
    if RUN_EN_CHECK and Ds.get("en") is not None and Qs.get("en") is not None:
        out["en"] = rank_lists(Qs["en"], Ds["en"], EN_T)
    if Ds.get("ood") is not None and Qs.get("ood") is not None:
        out["ood"] = rank_lists(Qs["ood"], Ds["ood"], OOD_T)
    return out

if REF_ZS_NOPFX not in RESULTS:
    r = eval_triplet({"vi": Q_test_np, "en": Q_en_np}, {"vi": D_test, "en": D_en})
    save_result(REF_ZS_NOPFX, vi=r["vi"], en=r.get("en"), vec_per_page=vec_per_page(D_test), embed_dim=EMB_DIM,
                encode_s_per_page=ENC_S_PER_PAGE, device=DEVICE_NAME, trained=False, group="zero-shot", n_seeds=1)
if REF_ZS not in RESULTS:
    r = eval_triplet({"vi": Q_test, "en": Q_en, "ood": Q_ood}, {"vi": D_test, "en": D_en, "ood": D_ood})
    save_result(REF_ZS, vi=r["vi"], en=r.get("en"), ood=r.get("ood"), vec_per_page=vec_per_page(D_test), embed_dim=EMB_DIM,
                encode_s_per_page=ENC_S_PER_PAGE, device=DEVICE_NAME, trained=False, group="zero-shot", n_seeds=1)
if REF_HIER not in RESULTS:
    r = eval_triplet({"vi": Q_test, "en": Q_en, "ood": Q_ood}, {"vi": HIER["test"], "en": HIER["en"], "ood": HIER.get("ood")})
    save_result(REF_HIER, vi=r["vi"], en=r.get("en"), ood=r.get("ood"), vec_per_page=vec_per_page(HIER["test"]), embed_dim=EMB_DIM,
                ms_per_page=HIER.get("ms_test"), device=f"CPU x{os.cpu_count()}", trained=False, group="zero-shot", n_seeds=1)

def _nd(name, key="vi"):
    v = RESULTS[name].get(key)
    return None if v is None else round(float(np.mean(seed_scores(v))), 2)

print("Kiểm chứng giao thức query (nDCG@5):")
for n in [REF_ZS_NOPFX, REF_ZS, REF_HIER]:
    print(f"  {n:55s} VI {_nd(n)} | EN {_nd(n, 'en')} | OOD {_nd(n, 'ood')}")
print("  Tham chiếu model card vidore/colqwen2.5-v0.2 trên DocVQA (ViDoRe v1): 63.22 "
      "(protocol MTEB/vidore-benchmark; bộ 500 câu của notebook có 451 câu khác nhau -> có thể lệch nhẹ)")
p_pfx = paired_randomization_p(pq_mean(RESULTS[REF_ZS]["vi"]), pq_mean(RESULTS[REF_ZS_NOPFX]["vi"]))
print(f"  Δ VI do tiền tố = {_nd(REF_ZS) - _nd(REF_ZS_NOPFX):+.2f} (randomization p = {fmt_p(p_pfx)})")

## 6. Hard negatives dùng chung (cùng file với notebook baseline)

In [ ]:
# =====================================================================================
# Hard negatives dùng chung cho MỌI mô hình train (notebook baseline và notebook v3 dùng cùng 1 file .npy)
# =====================================================================================
# ---- Hard negatives: khai thác bằng ColQwen2.5 zero-shot (+'Query:') trên trang KHÔNG nén (dùng chung mọi baseline) ----
VDOC_TR_T = np.asarray(tr_t, dtype=np.int64)
_hid = {}
VDOC_TR_H = np.asarray([_hid.setdefault(r["h"], len(_hid)) for r in sg_pages], dtype=np.int64)
HARD_P_PATH = f"{SAVE_DIR}/stage3_hardneg_prefix_k{HARD_NEG}_tr{N_TRAIN_PAGES}x{MAX_Q_PER_PAGE}.npy"
if os.path.exists(HARD_P_PATH):
    HARD_P = np.load(HARD_P_PATH)
else:
    HARD_P = mine_hard_negatives(Q_tr, D_tr, VDOC_TR_T, VDOC_TR_H, k=HARD_NEG)
    np.save(HARD_P_PATH, HARD_P)

## 7. Kiến trúc v3: `GridMultiVectorAdapter` (+ unit test trên CPU)
- **Nhánh tài liệu:** đặc trưng saliency trên lân cận 3×3 của lưới ảnh → trọng số patch; conv depthwise 2D + LayerNorm theo token (residual khởi đầu gần đồng nhất); gộp mỗi ô 2×4 → 2 vector bằng anchor bao phủ; refiner cross-attention 2 ← 8.
- **Nhánh câu hỏi:** self-attention hai chiều hạng thấp (giữ nguyên v2) + gate chỉ trên token nội dung, không còn tín hiệu biên câu.
- Unit test kiểm tra: ánh xạ ô 2×4 đúng hình học, output **không phụ thuộc trang khác trong batch**, chuẩn hoá L2, trung bình gate = 1, gradient tới mọi tham số.

In [ ]:
# =====================================================================================
# GridMultiVectorAdapter (v3) — chuẩn hoá lại DynamicMultiVectorAdapter (v2)
#  [Sửa B2] Cửa sổ 2×4 trên LƯỚI ẢNH THẬT (H'×W' token, thứ tự raster của Qwen2.5-VL) thay cho cửa sổ 1D 8 token
#           (v2 lệch ~4 token prompt, vắt qua hàng); token prompt/đặc biệt (~11) được gộp RIÊNG theo cửa sổ 1D.
#  [Sửa B3] Anchor học được: anchor 1 = token saliency cao nhất, anchor 2 = token XA anchor 1 nhất (bao phủ),
#           mọi token khác gán về anchor gần nhất, trộn theo trọng số saliency (v2: anchor cố định vị trí 0 và 4).
#  [Sửa B4] Bỏ GroupNorm trên chuỗi đã pad -> LayerNorm theo từng token + conv 2D trên vùng đã mask
#           => output của 1 trang KHÔNG phụ thuộc trang khác trong batch (có unit test).
#  [Sửa B5] Gate: không còn số 0 ở biên câu (f2/f3 biên = f4), gate chỉ áp lên token NỘI DUNG (bỏ 'Query:' và
#           10 token augmentation), thêm phạt tương quan vị trí–trọng số khi train.
# =====================================================================================
WIN_H, WIN_W = 2, 4          # cửa sổ 2×4 = 8 token ảnh liền kề -> giữ 2 vector (nén ×4)
PROMPT_WIN = 8               # token prompt gộp theo cửa sổ 1D 8 -> 2

def _ceil_to(x, k):
    return int(-(-int(x) // k) * k)

def pack_docs_grid(D_list, M_list, idx, device=None):
    """Đóng gói trang thành (lưới ảnh, mask, chuỗi prompt, mask). M_list[i] = (start, H', W') hoặc None (trang lỗi meta -> 1D)."""
    device = DEVICE if device is None else device
    items = [(D_list[i], M_list[i]) for i in idx]
    D = int(items[0][0].shape[-1])
    hs = [m[1] for _, m in items if m is not None] or [0]
    ws = [m[2] for _, m in items if m is not None] or [0]
    Hm, Wm = max(WIN_H, _ceil_to(max(hs), WIN_H)), max(WIN_W, _ceil_to(max(ws), WIN_W))
    B = len(items)
    grid = torch.zeros(B, Hm, Wm, D)
    gmask = torch.zeros(B, Hm, Wm, dtype=torch.bool)
    prompts = []
    for b, (E, m) in enumerate(items):
        E = E.float()
        if m is not None:
            s, h, w = m
            grid[b, :h, :w] = E[s:s + h * w].reshape(h, w, D)
            gmask[b, :h, :w] = True
            prompts.append(torch.cat([E[:s], E[s + h * w:]], dim=0))
        else:
            prompts.append(E)
    Lp = _ceil_to(max(1, max(p.shape[0] for p in prompts)), PROMPT_WIN)
    P = torch.zeros(B, Lp, D)
    pmask = torch.zeros(B, Lp, dtype=torch.bool)
    for b, p in enumerate(prompts):
        P[b, :p.shape[0]] = p
        pmask[b, :p.shape[0]] = True
    return grid.to(device), gmask.to(device), P.to(device), pmask.to(device)

def _to_windows_2d(t, B, H, W):
    """(B,H,W,...) -> (B, S=(H/2)*(W/4), 8, ...) theo ô 2×4, thứ tự raster các ô."""
    rest = t.shape[3:]
    t = t.reshape(B, H // WIN_H, WIN_H, W // WIN_W, WIN_W, *rest)
    perm = (0, 1, 3, 2, 4) + tuple(range(5, 5 + len(rest)))
    return t.permute(*perm).reshape(B, (H // WIN_H) * (W // WIN_W), WIN_H * WIN_W, *rest)

class GridMultiVectorAdapter(nn.Module):
    def __init__(self, dim=128, rank=32, dropout=0.1, compress=True, use_doc_adapter=True,
                 use_subword_attn=True, use_query_gate=True):
        super().__init__()
        self.dim, self.compress = dim, compress
        self.use_doc_adapter, self.use_subword_attn, self.use_query_gate = use_doc_adapter, use_subword_attn, use_query_gate
        # Nhánh tài liệu: conv 2D depthwise + LayerNorm theo token (thay GroupNorm) + bottleneck
        self.doc_dw = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
        self.doc_ln = nn.LayerNorm(dim)
        self.doc_down = nn.Linear(dim, rank, bias=False)
        self.doc_up = nn.Linear(rank, dim, bias=False)
        self.doc_drop = nn.Dropout(dropout)
        self.alpha_conv = nn.Parameter(torch.tensor(0.02))
        self.saliency_head = nn.Sequential(nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.GELU(), nn.Conv2d(16, 1, kernel_size=1))
        nn.init.normal_(self.saliency_head[-1].weight, std=0.02)
        nn.init.zeros_(self.saliency_head[-1].bias)
        self.saliency_temp = nn.Parameter(torch.tensor(0.30))
        # Refiner cross-attention (2 anchor <- 8 token gốc của ô)
        self.merge_norm = nn.LayerNorm(dim)
        self.merge_q = nn.Linear(dim, rank, bias=False)
        self.merge_k = nn.Linear(dim, rank, bias=False)
        self.merge_v = nn.Linear(dim, rank, bias=False)
        self.merge_out = nn.Linear(rank, dim, bias=False)
        self.alpha_merge = nn.Parameter(torch.tensor(0.04))
        # Nhánh câu hỏi
        self.sub_norm = nn.LayerNorm(dim)
        self.sub_q = nn.Linear(dim, rank, bias=False)
        self.sub_k = nn.Linear(dim, rank, bias=False)
        self.sub_v = nn.Linear(dim, rank, bias=False)
        self.sub_out = nn.Linear(rank, dim, bias=False)
        self.alpha_sub = nn.Parameter(torch.tensor(0.04))
        self.gate_mlp = nn.Sequential(nn.Linear(dim + 5, 48), nn.LayerNorm(48), nn.GELU(), nn.Dropout(dropout), nn.Linear(48, 1))
        nn.init.normal_(self.gate_mlp[-1].weight, std=0.01)
        nn.init.zeros_(self.gate_mlp[-1].bias)
        self.gate_scale = nn.Parameter(torch.tensor(0.30))

    # ---------------- Nhánh tài liệu ----------------
    def _doc_adapt(self, x, m):
        """x: (B,H,W,D) đã chuẩn hoá & mask; m: (B,H,W) bool. Trả về x', trọng số patch, saliency logit."""
        mf = m.float()
        if not self.use_doc_adapter:
            return x, mf, torch.zeros_like(mf)
        xc = x.permute(0, 3, 1, 2)
        mc = mf.unsqueeze(1)
        n_valid = mf.sum(dim=(1, 2)).clamp(min=1.0)
        s3 = F.avg_pool2d(xc, 3, 1, 1) * 9.0
        c3 = F.avg_pool2d(mc, 3, 1, 1) * 9.0
        diff_local = (xc - s3 / c3.clamp(min=1.0)).norm(dim=1) * mf
        page_mean = x.sum(dim=(1, 2)) / n_valid[:, None]
        diff_glob = (x - page_mean[:, None, None, :]).norm(dim=-1) * mf
        max_ch = x.abs().amax(dim=-1) * mf
        feats = torch.stack([diff_local, diff_glob, max_ch], dim=1)
        mu = (feats * mc).sum(dim=(2, 3), keepdim=True) / n_valid[:, None, None, None]
        var = (((feats - mu) ** 2) * mc).sum(dim=(2, 3), keepdim=True) / n_valid[:, None, None, None]
        feats = ((feats - mu) / (var.sqrt() + 1e-5)) * mc
        sal = self.saliency_head(feats).squeeze(1) * mf
        w = torch.exp(torch.clamp(self.saliency_temp, 0.05, 0.8) * torch.tanh(sal)) * mf
        h = self.doc_dw(xc).permute(0, 2, 3, 1)
        h = self.doc_up(self.doc_drop(F.gelu(self.doc_down(F.gelu(self.doc_ln(h))))))
        x = F.normalize(x + 0.15 * torch.tanh(self.alpha_conv) * h, p=2, dim=-1) * mf.unsqueeze(-1)
        return x, w, sal

    @staticmethod
    def _coverage_merge(wx, ww, wm, wsal):
        """wx (B,S,8,D) | ww,wm,wsal (B,S,8). Mỗi ô -> tối đa 2 vector: anchor saliency cao nhất + anchor xa nhất."""
        NEG = -1e4
        a1 = wsal.masked_fill(~wm, NEG).argmax(dim=-1)
        x1 = torch.gather(wx, 2, a1[..., None, None].expand(-1, -1, 1, wx.shape[-1])).squeeze(2)
        cos1 = (wx * x1.unsqueeze(2)).sum(-1)
        sc2 = ((1.0 - cos1) + 0.1 * torch.tanh(wsal)).masked_fill(~wm, NEG)
        sc2 = sc2.scatter(-1, a1.unsqueeze(-1), NEG)
        a2 = sc2.argmax(dim=-1)
        two = wm.sum(-1) >= 2
        x2 = torch.gather(wx, 2, a2[..., None, None].expand(-1, -1, 1, wx.shape[-1])).squeeze(2)
        cos2 = (wx * x2.unsqueeze(2)).sum(-1).masked_fill(~two.unsqueeze(-1), -2.0)
        to2 = cos2 > cos1
        wv = ww * wm.float()
        w1, w2 = wv * (~to2).float(), wv * to2.float()
        s1 = (w1.unsqueeze(-1) * wx).sum(2)
        s2 = (w2.unsqueeze(-1) * wx).sum(2)
        m1 = wm.any(-1) & (w1.sum(-1) > 1e-6)
        m2 = two & (w2.sum(-1) > 1e-6)
        seeds = torch.stack([F.normalize(s1, p=2, dim=-1) * m1.unsqueeze(-1).float(),
                             F.normalize(s2, p=2, dim=-1) * m2.unsqueeze(-1).float()], dim=2)
        return seeds, torch.stack([m1, m2], dim=-1)

    def _refine(self, seeds, smask, wx, wm, wsal):
        sn, qn = self.merge_norm(wx), self.merge_norm(seeds)
        Q, K, V = self.merge_q(qn), self.merge_k(sn), self.merge_v(sn)
        logits = torch.einsum("bsqr,bskr->bsqk", Q, K) / (Q.shape[-1] ** 0.5) + 0.3 * torch.tanh(wsal).unsqueeze(2)
        logits = logits.masked_fill(~wm.unsqueeze(2), -1e4)
        attn = F.softmax(logits, dim=-1) * wm.float().unsqueeze(2)
        out = self.merge_out(torch.einsum("bsqk,bskr->bsqr", attn, V))
        return F.normalize(seeds + 0.25 * torch.tanh(self.alpha_merge) * out, p=2, dim=-1) * smask.float().unsqueeze(-1)

    def forward_docs(self, grid, gmask, P, pmask):
        B, H, W, D = grid.shape
        x = F.normalize(grid.float(), p=2, dim=-1) * gmask.float().unsqueeze(-1)
        x, w, sal = self._doc_adapt(x, gmask)
        p = F.normalize(P.float(), p=2, dim=-1) * pmask.float().unsqueeze(-1)
        if not self.compress:
            out = torch.cat([x.reshape(B, H * W, D), p], dim=1)
            return out, torch.cat([gmask.reshape(B, H * W), pmask], dim=1)
        wx, ww = _to_windows_2d(x, B, H, W), _to_windows_2d(w, B, H, W)
        wm, wsal = _to_windows_2d(gmask, B, H, W), _to_windows_2d(sal, B, H, W)
        seeds, smask = self._coverage_merge(wx, ww, wm, wsal)
        if self.use_doc_adapter:
            seeds = self._refine(seeds, smask, wx, wm, wsal)
        Lp = P.shape[1]
        px = p.reshape(B, Lp // PROMPT_WIN, PROMPT_WIN, D)
        pm = pmask.reshape(B, Lp // PROMPT_WIN, PROMPT_WIN)
        pseeds, psmask = self._coverage_merge(px, pm.float(), pm, torch.zeros_like(pm, dtype=torch.float32))
        out = torch.cat([seeds.reshape(B, -1, D), pseeds.reshape(B, -1, D)], dim=1)
        return out, torch.cat([smask.reshape(B, -1), psmask.reshape(B, -1)], dim=1)

    # ---------------- Nhánh câu hỏi ----------------
    def forward_queries(self, E_q, q_mask, c_mask=None):
        qm = q_mask.bool()
        vf = qm.float()
        q0 = F.normalize(E_q.float(), p=2, dim=-1) * vf.unsqueeze(-1)
        if self.use_subword_attn:
            qn = self.sub_norm(q0)
            Q, K, V = self.sub_q(qn), self.sub_k(qn), self.sub_v(qn)
            sc = (torch.einsum("bqr,bkr->bqk", Q, K) / (Q.shape[-1] ** 0.5)).masked_fill(~qm.unsqueeze(1), -1e4)
            att = F.softmax(sc, dim=-1) * vf.unsqueeze(-1)
            q = F.normalize(q0 + 0.20 * torch.tanh(self.alpha_sub) * self.sub_out(torch.einsum("bqk,bkr->bqr", att, V)), p=2, dim=-1)
            q = q * vf.unsqueeze(-1)
        else:
            q = q0
        if not self.use_query_gate:
            return q, None, torch.zeros_like(vf), q0
        B, L, _ = q.shape
        cm = qm if c_mask is None else (c_mask.bool() & qm)
        n_valid = vf.sum(-1, keepdim=True).clamp(min=1.0)
        q_glob = F.normalize(q.sum(1, keepdim=True) / n_valid.unsqueeze(-1), p=2, dim=-1)
        f1 = 1.0 - (q * q_glob).sum(-1)
        sim = torch.einsum("bqd,bkd->bqk", q, q)
        peer = qm.unsqueeze(1) & qm.unsqueeze(2) & ~torch.eye(L, dtype=torch.bool, device=q.device).unsqueeze(0)
        f4 = sim.masked_fill(~peer, 0.0).sum(-1) / (n_valid - 1.0).clamp(min=1.0)
        f5 = sim.masked_fill(~peer, -1.0).amax(-1).clamp(min=0.0)
        prev_ok = torch.zeros_like(qm); prev_ok[:, 1:] = qm[:, :-1] & qm[:, 1:]
        next_ok = torch.zeros_like(qm); next_ok[:, :-1] = qm[:, 1:] & qm[:, :-1]
        f2 = torch.zeros_like(f1); f2[:, 1:] = (q[:, 1:] * q[:, :-1]).sum(-1)
        f3 = torch.zeros_like(f1); f3[:, :-1] = (q[:, :-1] * q[:, 1:]).sum(-1)
        f2 = torch.where(prev_ok, f2, f4)   # [Sửa B5] biên câu không còn mang giá trị 0 đặc trưng vị trí
        f3 = torch.where(next_ok, f3, f4)
        g_in = torch.cat([q, torch.stack([f1, f2, f3, f4, f5], dim=-1)], dim=-1) * vf.unsqueeze(-1)
        delta = torch.tanh(self.gate_mlp(g_in)).squeeze(-1) * cm.float()
        raw = (1.0 + torch.clamp(self.gate_scale, 0.05, 0.50) * delta) * vf
        gates = raw * (n_valid / (raw.sum(-1, keepdim=True) + 1e-6))
        return q, gates, delta, q0

def gate_position_penalty(delta, c_mask):
    """Trung bình bình phương hệ số tương quan Pearson giữa vị trí tương đối (trên token nội dung) và gate_delta."""
    cm = c_mask.float()
    n = cm.sum(-1)
    ok = n >= 3
    if not ok.any():
        return delta.sum() * 0.0
    pos = (torch.cumsum(cm, dim=-1) - 1.0) / (n.unsqueeze(-1) - 1.0).clamp(min=1.0)
    mp = (pos * cm).sum(-1) / n.clamp(min=1.0)
    md = (delta * cm).sum(-1) / n.clamp(min=1.0)
    dp, dd = (pos - mp.unsqueeze(-1)) * cm, (delta - md.unsqueeze(-1)) * cm
    corr = (dp * dd).sum(-1) / ((dp.pow(2).sum(-1) * dd.pow(2).sum(-1)).sqrt() + 1e-6)
    return corr[ok].pow(2).mean()

def v3_doc_count(m, L):
    """Số vector đầu ra (cận trên, khi mọi ô có ≥2 token) — để so Hier cùng ngân sách."""
    if m is None:
        return 2 * (_ceil_to(L, PROMPT_WIN) // PROMPT_WIN)
    s, h, w = m
    n_prompt = L - h * w
    return 2 * (_ceil_to(h, WIN_H) // WIN_H) * (_ceil_to(w, WIN_W) // WIN_W) + 2 * (_ceil_to(max(1, n_prompt), PROMPT_WIN) // PROMPT_WIN)

# ---------------- Unit tests (CPU, dữ liệu giả) ----------------
def _unit_tests_v3():
    g = torch.Generator().manual_seed(0)
    Dm = 16
    def page(h, w, n_pre=4, n_post=7):
        L = n_pre + h * w + n_post
        return F.normalize(torch.randn(L, Dm, generator=g), dim=-1), (n_pre, h, w)
    pages = [page(5, 7), page(3, 9), page(6, 4)]
    D_l, M_l = [p[0] for p in pages], [p[1] for p in pages]
    # (1) ánh xạ ô 2×4 đúng hình học: mã hoá (hàng, cột) vào giá trị
    Hh, Ww = 4, 8
    code = torch.zeros(1, Hh, Ww, 2)
    for r in range(Hh):
        for c in range(Ww):
            code[0, r, c] = torch.tensor([r, c], dtype=torch.float32)
    win = _to_windows_2d(code, 1, Hh, Ww)
    assert win.shape == (1, 4, 8, 2)
    assert set(map(tuple, win[0, 1].int().tolist())) == {(r, c) for r in (0, 1) for c in range(4, 8)}
    assert set(map(tuple, win[0, 2].int().tolist())) == {(r, c) for r in (2, 3) for c in range(0, 4)}
    for compress in (True, False):
        net = GridMultiVectorAdapter(dim=Dm, rank=8, compress=compress).eval()
        with torch.no_grad():
            # (2) bất biến với padding/batch: trang 0 một mình vs chung batch với trang lớn hơn
            out_a, m_a = net.forward_docs(*pack_docs_grid(D_l, M_l, [0], device="cpu"))
            out_b, m_b = net.forward_docs(*pack_docs_grid(D_l, M_l, [2, 0, 1], device="cpu"))
            va, vb = out_a[0][m_a[0]], out_b[1][m_b[1]]
            assert va.shape == vb.shape, (va.shape, vb.shape)
            assert torch.allclose(va, vb, atol=1e-5), "Output phụ thuộc batch/padding!"
            assert torch.allclose(va.norm(dim=-1), torch.ones(len(va)), atol=1e-4)
            if compress:
                assert len(va) <= v3_doc_count(M_l[0], len(D_l[0]))
                assert v3_doc_count((4, 32, 23), 747) == 196  # trang V-Doc điển hình: 747 -> 196 vec (×3,8)
            else:
                assert len(va) == len(D_l[0])
    # (3) gate: trung bình = 1 trên token hợp lệ; token không phải nội dung giữ delta = 0
    net = GridMultiVectorAdapter(dim=Dm, rank=8).eval()
    with torch.no_grad():
        nn.init.normal_(net.gate_mlp[-1].weight, std=1.0)
        E = torch.randn(2, 9, Dm, generator=g)
        qm = torch.tensor([[1] * 9, [1] * 6 + [0] * 3], dtype=torch.bool)
        cm = qm.clone(); cm[:, :2] = False
        q, gates, delta, _ = net.forward_queries(E, qm, cm)
        assert torch.allclose((gates * qm).sum(-1) / qm.sum(-1), torch.ones(2), atol=1e-4)
        assert torch.all(delta[~cm] == 0)
        pen = gate_position_penalty(delta, cm)
        assert 0.0 <= float(pen) <= 1.0 + 1e-6
    # (4) gradient chảy qua mọi khối được bật
    net = GridMultiVectorAdapter(dim=Dm, rank=8).train()
    grid, gm, P, pm = pack_docs_grid(D_l, M_l, [0, 1, 2], device="cpu")
    d, dm = net.forward_docs(grid, gm, P, pm)
    q, gates, delta, q0 = net.forward_queries(E, qm, cm)
    s = score_masked_maxsim(q, qm, d, dm, gates=gates, q_chunk_size=2, d_chunk_size=2)
    (s.sum() + gate_position_penalty(delta, cm)).backward()
    for n_, p_ in net.named_parameters():
        assert p_.grad is not None, f"Không có gradient: {n_}"
    print(f"✓ Unit test GridMultiVectorAdapter OK | tham số: {sum(p.numel() for p in GridMultiVectorAdapter(dim=128, rank=32).parameters()):,}")

_unit_tests_v3()

## 8. Dữ liệu train bổ sung (đã sửa): EN replay & 5CD-AI `vi_image`
Cùng 5 lớp lọc leakage như Giai đoạn 2; audit ghi rõ cột ảnh thực sự được dùng.

In [ ]:
# =====================================================================================
# Dữ liệu train bổ sung (đã sửa) — cả hai đều CHỈ dùng cho train, không bao giờ dùng để val/test
#  (a) English replay: vidore/colpali_train_set (public, MIT) — chính phân phối mà ColQwen2.5 đã học -> chống quên EN
#  (b) 5CD-AI Viet-colpali (gated): [Sửa B7] dùng cột `vi_image` (ảnh đã dịch & render lại tiếng Việt) thay vì `image`
#      (ảnh gốc tiếng Anh). Ghi lại cột ảnh thực sự dùng vào audit.
#  Cùng 5 lớp lọc leakage như Giai đoạn 2 (bỏ docvqa, lọc query trùng, hash ảnh cấm, cap theo nguồn, assert).
# =====================================================================================
def filter_stream_v3(stream_iter, forbidden_hashes, forbidden_q_norm, max_pages, max_per_source, max_scan,
                     save_img_dir, image_keys, file_prefix):
    stats = {"scanned": 0, "accepted": 0, "rej_docvqa": 0, "rej_bad_q": 0, "rej_q_leak": 0,
             "rej_source_cap": 0, "rej_bad_img": 0, "rej_hash_leak": 0, "rej_dup": 0}
    used_key = Counter()
    accepted, source_counts, seen_h = [], Counter(), set()
    os.makedirs(save_img_dir, exist_ok=True)
    for row in stream_iter:
        if stats["scanned"] >= max_scan or stats["accepted"] >= max_pages:
            break
        stats["scanned"] += 1
        src = str(row.get("source") or "unknown").strip() or "unknown"
        if "docvqa" in src.lower():
            stats["rej_docvqa"] += 1
            continue
        q = row.get("query")
        if not isinstance(q, str) or len(q.strip()) < 12:
            stats["rej_bad_q"] += 1
            continue
        if norm_q(q) in forbidden_q_norm:
            stats["rej_q_leak"] += 1
            continue
        if source_counts[src] >= max_per_source:
            stats["rej_source_cap"] += 1
            continue
        im, key = None, None
        for k in image_keys:
            if row.get(k) is not None:
                im, key = row[k], k
                break
        if im is None:
            stats["rej_bad_img"] += 1
            continue
        try:
            rgb = im.convert("RGB") if isinstance(im, Image.Image) else Image.open(im).convert("RGB")
        except Exception:
            stats["rej_bad_img"] += 1
            continue
        h = img_hash_pil(rgb)
        if h in forbidden_hashes:
            stats["rej_hash_leak"] += 1
            continue
        if h in seen_h:
            stats["rej_dup"] += 1
            continue
        seen_h.add(h)
        source_counts[src] += 1
        used_key[key] += 1
        out_p = os.path.join(save_img_dir, f"{file_prefix}_{stats['accepted']:05d}.jpg")
        rgb.save(out_p, quality=95)
        accepted.append({"query": q.strip(), "source": src, "h": h, "path": out_p, "image_key": key, "size": list(rgb.size)})
        stats["accepted"] += 1
    stats["image_key_used"] = dict(used_key)
    return accepted, stats, dict(source_counts)

def load_or_build_extra(tag, repo, image_keys, n_pages, max_per_source, needs_token):
    audit_p = f"{SAVE_DIR}/stage3_{tag}_audit_{n_pages}.json"
    emb_p = f"{SAVE_DIR}/stage3_{tag}_emb_{n_pages}.pt"
    img_dir = f"{DATA_DIR}/stage3_{tag}_{n_pages}"
    if needs_token and not os.environ.get("HF_TOKEN"):
        print(f"⚠ Bỏ qua {tag}: repo gated cần HF_TOKEN (Colab Secrets hoặc dán vào ô cấu hình).")
        return None
    audit = json.load(open(audit_p, encoding="utf-8")) if os.path.exists(audit_p) else None
    if audit is not None and not os.path.exists(emb_p) and not all(os.path.exists(r["path"]) for r in audit["pages"]):
        audit = None  # chưa có embedding mà ảnh (đĩa runtime) đã mất -> stream lại
    if audit is None:
        print(f">>> Stream & lọc {repo} ...")
        ds = load_dataset(repo, split="train", streaming=True)
        pages, st, srcs = filter_stream_v3(ds, forbidden_hashes, forbidden_q_norm, n_pages, max_per_source,
                                          MAX_5CD_SCAN, img_dir, image_keys, tag)
        assert {r["h"] for r in pages}.isdisjoint(forbidden_hashes), f"Rò rỉ hash ở {tag}!"
        assert not ({norm_q(r["query"]) for r in pages} & forbidden_q_norm), f"Rò rỉ query ở {tag}!"
        audit = {"stats": st, "source_counts": srcs, "pages": pages}
        with open(audit_p, "w", encoding="utf-8") as f:
            json.dump(audit, f, ensure_ascii=False)
    print(f"Audit {tag}:", audit["stats"], "| nguồn:", audit["source_counts"])
    if not os.path.exists(emb_p):
        m = get_qwen_model()
        D_x = encode(m, PROC_Q, [r["path"] for r in audit["pages"]], True, bs=QWEN_IMG_BS)
        Q_x, _, C_x = build_query_pack(m, PROC_QP, [r["query"] for r in audit["pages"]], bs=32, name=tag)
        torch.save({"D": D_x, "Q": Q_x, "C": C_x}, emb_p)
    pk = torch.load(emb_p, map_location="cpu", weights_only=False)
    return {"pages": audit["pages"], "D": pk["D"], "Q": pk["Q"], "C": pk["C"], "stats": audit["stats"]}

EXTRA_REPLAY = EXTRA_5CDVI = None
if RUN_EN_REPLAY:
    try:
        EXTRA_REPLAY = load_or_build_extra("replay_en", "vidore/colpali_train_set", ("image",), N_REPLAY, max(1, N_REPLAY // 4), False)
    except Exception as exc:
        print("⚠ Bỏ qua English replay:", exc)
        traceback.print_exc()
if RUN_5CD_VI_MIX:
    try:
        EXTRA_5CDVI = load_or_build_extra("5cd_vi_image", "5CD-AI/Viet-colpali_train_set-Gemini", ("vi_image", "image"),
                                          N_5CD_PAGES, MAX_PER_5CD_SOURCE, True)
        if EXTRA_5CDVI is not None and EXTRA_5CDVI["stats"].get("image_key_used", {}).get("vi_image", 0) == 0:
            print("⚠ Không tìm thấy cột vi_image -> nhánh này đang dùng ảnh gốc (giống v2); kết quả Mix-vi KHÔNG có giá trị sửa lỗi.")
    except Exception as exc:
        print("⚠ Bỏ qua 5CD-AI vi_image:", exc)
        traceback.print_exc()
release_qwen_model()

## 9. Meta lưới ảnh từng trang + chẩn đoán gần-trùng test ↔ train
Lấy (H', W') từ kích thước ảnh qua `processor.get_n_patches` (không cần chạy lại model), kiểm tra khớp độ dài chuỗi embedding; trang nào không khớp được xử lý 1D (in ra số lượng). Tính độ giống test→train gần nhất để tách **nửa test xa train** làm thước đo tổng quát hoá bổ sung.

In [ ]:
# =====================================================================================
# Meta lưới ảnh cho từng trang: (vị trí token ảnh đầu tiên, H', W') — KHÔNG cần chạy lại model
#   Chuỗi vector của 1 trang ColQwen2.5 = [4 token prompt] + [H'×W' token ảnh, thứ tự raster] + [7 token prompt]
#   H', W' = lưới sau khi gộp 2×2 patch 14px (mỗi token = ô 28×28 px của ảnh đã resize động)
# =====================================================================================
def _probe_layout():
    b = PROC_Q.process_images([load_img(TEST_DOCS[0])])
    ids = b["input_ids"][0]
    img_id = PROC_Q.tokenizer.convert_tokens_to_ids(PROC_Q.image_token)
    pos = (ids == img_id).nonzero().flatten()
    start, n_img = int(pos[0]), int(len(pos))
    assert torch.equal(pos, torch.arange(start, start + n_img)), "Token ảnh không liền nhau!"
    ms = int(getattr(PROC_Q.image_processor, "merge_size", 2))
    t, h, w = [int(x) for x in b["image_grid_thw"][0].tolist()]
    assert n_img == t * (h // ms) * (w // ms)
    return start, len(ids) - n_img, ms

META_START, META_NPROMPT, META_MERGE = _probe_layout()
print(f"Bố cục chuỗi trang: {META_START} token prompt trước ảnh, tổng {META_NPROMPT} token không phải ảnh, merge {META_MERGE}×{META_MERGE}")

def _grid_from_size(path_or_size):
    if isinstance(path_or_size, (list, tuple)):
        size = tuple(path_or_size)
    else:
        with Image.open(path_or_size) as im:
            size = im.size
    nx, ny = PROC_Q.get_n_patches(size, spatial_merge_size=META_MERGE)
    return int(ny), int(nx)

def _grid_from_processor(path):
    t, h, w = [int(x) for x in PROC_Q.process_images([load_img(path)])["image_grid_thw"][0].tolist()]
    return h // META_MERGE, w // META_MERGE

def build_meta(paths, D, name, sizes=None):
    M, n_fallback, n_bad = [], 0, 0
    for i, (p, E) in enumerate(zip(paths, D)):
        L = int(E.shape[0])
        hw = None
        try:
            hw = _grid_from_size(sizes[i] if sizes is not None else p)
        except Exception:
            hw = None
        if hw is None or hw[0] * hw[1] + META_NPROMPT != L:
            n_fallback += 1
            try:
                hw = _grid_from_processor(p)
            except Exception:
                hw = None
        if hw is not None and hw[0] * hw[1] + META_NPROMPT == L:
            M.append((META_START, hw[0], hw[1]))
        else:
            M.append(None)
            n_bad += 1
    print(f"  meta [{name}]: {len(M)} trang | dùng processor dự phòng {n_fallback} | không khớp (xử lý 1D) {n_bad}")
    return M

META_CACHE = f"{SAVE_DIR}/stage3_page_meta_s{N_SHARDS}_tr{N_TRAIN_PAGES}x{MAX_Q_PER_PAGE}_v{N_VAL}.pt"
META = torch.load(META_CACHE, map_location="cpu", weights_only=False) if os.path.exists(META_CACHE) else {}
_sets = {"tr": ([r["path"] for r in sg_pages], D_tr), "val": ([r["path"] for r in val_pages], D_val),
         "test": (TEST_DOCS, D_test), "en": (EN_DOCS, D_en)}
if D_ood is not None:
    _sets["ood"] = (OOD_DOCS, D_ood)
_sizes = {}
for _k, _ex in [("replay_en", EXTRA_REPLAY), ("5cd_vi_image", EXTRA_5CDVI)]:
    if _ex is not None:
        _sets[_k] = ([r["path"] for r in _ex["pages"]], _ex["D"])
        _sizes[_k] = [r["size"] for r in _ex["pages"]]
_ch = False
for k, (paths, D) in _sets.items():
    if k not in META or len(META[k]) != len(D):
        if k in _sizes:
            META[k] = build_meta(paths, D, k, sizes=_sizes[k])
        elif not all(os.path.exists(p) for p in paths):
            print(f"  meta [{k}]: thiếu file ảnh trên runtime -> xử lý 1D cho tập này")
            META[k] = [None] * len(D)
        else:
            META[k] = build_meta(paths, D, k)
        _ch = True
if _ch:
    torch.save(META, META_CACHE)
M_tr, M_val, M_test, M_en = META["tr"], META["val"], META["test"], META["en"]
M_ood = META.get("ood")
_hs = [m[1] for m in M_test if m]; _ws = [m[2] for m in M_test if m]
print(f"Lưới test: H' trung vị {int(np.median(_hs))}, W' trung vị {int(np.median(_ws))} | "
      f"tỉ lệ trang W' chia hết cho 4: {np.mean([w % 4 == 0 for w in _ws]):.2f} | "
      f"v3 dự kiến {np.mean([v3_doc_count(m, len(E)) for m, E in zip(M_test, D_test)]):.1f} vec/trang")

# ---- Chẩn đoán gần-trùng giữa test và train (giới hạn của chia theo khối id) ----
def page_vecs(D, M):
    out = []
    for E, m in zip(D, M):
        x = E.float()
        if m is not None:
            x = x[m[0]:m[0] + m[1] * m[2]]
        out.append(F.normalize(x.mean(0), dim=-1))
    return torch.stack(out)

_Vtr, _Vte = page_vecs(D_tr, M_tr), page_vecs(D_test, M_test)
TEST_MAXSIM_TRAIN = (_Vte @ _Vtr.T).max(dim=1).values.numpy()
FAR_PAGE = TEST_MAXSIM_TRAIN <= np.median(TEST_MAXSIM_TRAIN)
FAR_MASK = FAR_PAGE[np.asarray(TEST_T)]
print("Độ giống (cosine, vector trung bình trang) test→train gần nhất: "
      + ", ".join(f"q{q}={np.quantile(TEST_MAXSIM_TRAIN, q / 100):.3f}" for q in (10, 50, 90, 99))
      + f" | nửa 'xa train' = {FAR_MASK.sum()} câu hỏi")

## 10. Huấn luyện & đánh giá v3 (3 seed) + Hier cùng ngân sách + ablation
Thứ tự: `V3-A` (chưa train) → Hier cùng ngân sách → `V3-Full` (teacher) → `V3-C` (+KD) → `V3-C −KD` → ablation → biến thể dữ liệu (EN replay, 5CD `vi_image`). Mỗi biến thể lưu ngay vào JSON, chạy lại sẽ bỏ qua phần đã xong.

In [ ]:
# =====================================================================================
# Huấn luyện & đánh giá v3 (cùng lịch train Giai đoạn 2: AdamW 5e-4, warmup 1 + cosine 12 epoch, BQ=32,
# 7 hard negatives, mask false-negative theo hash, chọn epoch theo val nDCG@5, 3 seed)
#  [Sửa B6] KD có nghĩa: teacher = V3-Full CÙNG seed (adapter KHÔNG nén, mạnh hơn student), cùng nhiệt độ với CE,
#           KHÔNG nhân τ² (để gradient KD cùng bậc với CE) — thay cho "self-KD" từ backbone yếu hơn ở v2.
# =====================================================================================
def pad_bool(arrs, L):
    out = torch.zeros(len(arrs), L, dtype=torch.bool)
    for i, a in enumerate(arrs):
        a = torch.as_tensor(np.asarray(a, dtype=bool))
        out[i, :min(L, len(a))] = a[:L]
    return out

@torch.no_grad()
def encode_docs_v3(net, D, M, doc_bs=32):
    net.eval()
    out_all = []
    for i in range(0, len(D), doc_bs):
        idx = list(range(i, min(i + doc_bs, len(D))))
        out, om = net.forward_docs(*pack_docs_grid(D, M, idx))
        out, om = out.to("cpu", torch.bfloat16), om.cpu()
        out_all += [x[m] for x, m in zip(out, om)]
    return out_all

@torch.no_grad()
def encode_queries_v3(net, Q, C, q_bs=64):
    net.eval()
    Qo, Go = [], []
    for i in range(0, len(Q), q_bs):
        q_pad, q_mask = _pad_with_mask(Q[i:i + q_bs])
        c_pad = pad_bool(C[i:i + q_bs], q_pad.shape[1])
        q, g, _, _ = net.forward_queries(q_pad.to(DEVICE), q_mask.to(DEVICE), c_pad.to(DEVICE))
        q = q.to("cpu", torch.bfloat16)
        g = g.float().cpu() if g is not None else None
        for j in range(len(q)):
            m = q_mask[j]
            Qo.append(q[j][m])
            if g is not None:
                Go.append(g[j][m])
    return Qo, (Go if net.use_query_gate else None)

def eval_v3(net, Q, C, D, M, T):
    Dc = encode_docs_v3(net, D, M)
    Qo, Go = encode_queries_v3(net, Q, C)
    return ranks_from_scores(score_ragged_maxsim(Qo, Dc, G_list=Go), T), Dc

def _v3_scores(net, q_pad, q_mask, c_pad, packed):
    d, dm = net.forward_docs(*packed)
    q, gates, delta, q0 = net.forward_queries(q_pad, q_mask, c_pad)
    qcs = SCORE_CHUNK if net.compress else 4
    return score_masked_maxsim(q, q_mask, d, dm, gates=gates, q_chunk_size=qcs, d_chunk_size=DOC_CHUNK), q, delta, q0

def train_v3(spec, data, seed, teacher=None):
    torch.manual_seed(seed)
    gen = np.random.default_rng(seed)
    net = GridMultiVectorAdapter(dim=EMB_DIM, rank=32, dropout=0.1, compress=spec["compress"], use_doc_adapter=spec["doc"],
                                 use_subword_attn=spec["sub"], use_query_gate=spec["gate"]).to(DEVICE)
    Q, C, D, M = data["Q"], data["C"], data["D"], data["M"]
    T, Hh, HARD = data["T"], data["H"], data["HARD"]
    Nq = len(Q)
    spe = int(np.ceil(Nq / BQ))
    opt = torch.optim.AdamW(net.parameters(), lr=LR_ADAPTER, weight_decay=WEIGHT_DECAY)
    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=spe * WARMUP_EPOCHS, num_training_steps=spe * EPOCHS)
    best, best_state = -1.0, None
    for ep in range(EPOCHS):
        net.train()
        perm = gen.permutation(Nq)
        ep_loss, n_steps = 0.0, 0
        for i in range(0, Nq, BQ):
            idx = perm[i:i + BQ]
            pos = T[idx]
            cand = np.unique(np.concatenate([pos, HARD[idx].reshape(-1)]))
            tgt = torch.as_tensor(np.searchsorted(cand, pos), dtype=torch.long, device=DEVICE)
            fn = torch.as_tensor((Hh[cand][None, :] == Hh[pos][:, None]) & (cand[None, :] != pos[:, None]), device=DEVICE)
            q_pad, q_mask = _pad_with_mask([Q[j] for j in idx])
            c_pad = pad_bool([C[j] for j in idx], q_pad.shape[1]).to(DEVICE)
            q_pad, q_mask = q_pad.to(DEVICE), q_mask.to(DEVICE)
            packed = pack_docs_grid(D, M, cand)
            s, q, delta, q0 = _v3_scores(net, q_pad, q_mask, c_pad, packed)
            s = s.masked_fill(fn, -1e4)
            loss = F.cross_entropy(s / TEMP_CE, tgt)
            if teacher is not None and spec["kd"]:
                with torch.no_grad():
                    t, _, _, _ = _v3_scores(teacher, q_pad, q_mask, c_pad, packed)
                    t = t.masked_fill(fn, -1e4)
                loss = loss + KD_W_V3 * F.kl_div(F.log_softmax(s / TEMP_CE, -1), F.softmax(t / TEMP_CE, -1), reduction="batchmean")
            if spec["sub"]:
                loss = loss + ANCHOR_W * (1.0 - (q * q0).sum(-1))[q_mask].mean()
            if spec["gate"]:
                loss = loss + 0.005 * delta.abs().mean() + POS_W_V3 * gate_position_penalty(delta, c_pad & q_mask)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            sched.step()
            ep_loss += float(loss.item()); n_steps += 1
        nd = float(metrics(eval_v3(net, data["Qv"], data["Cv"], data["Dv"], data["Mv"], VAL_T)[0])["nDCG@5"])
        if nd > best:
            best, best_state = nd, copy.deepcopy(net.state_dict())
        print(f"  epoch {ep + 1}/{EPOCHS}  loss {ep_loss / max(1, n_steps):.4f}  val nDCG@5 {nd:.1f}")
    net.load_state_dict(best_state)
    return net.eval()

# ---- Tập train: V-Doc, V-Doc + EN replay, V-Doc + 5CD vi_image ----
def make_train_data(extra=None):
    Q, C, D, M = list(Q_tr), list(C_tr), list(D_tr), list(M_tr)
    T, hashes = VDOC_TR_T.copy(), [r["h"] for r in sg_pages]
    HARD = HARD_P
    if extra is not None:
        tag, ex = extra
        n0 = len(D)
        Q += ex["Q"]; C += ex["C"]; D += ex["D"]; M += META[tag]
        T = np.concatenate([T, np.arange(n0, n0 + len(ex["D"]), dtype=np.int64)])
        hashes += [r["h"] for r in ex["pages"]]
        hp = f"{SAVE_DIR}/stage3_hardneg_prefix_k{HARD_NEG}_{tag}_{len(ex['D'])}.npy"
    hid = {}
    Hh = np.asarray([hid.setdefault(h, len(hid)) for h in hashes], dtype=np.int64)
    if extra is not None:
        if os.path.exists(hp):
            HARD = np.load(hp)
        else:
            HARD = mine_hard_negatives(Q, D, T, Hh, k=HARD_NEG)
            np.save(hp, HARD)
    return {"Q": Q, "C": C, "D": D, "M": M, "T": T, "H": Hh, "HARD": HARD,
            "Qv": Q_val, "Cv": C_val, "Dv": D_val, "Mv": M_val}

V3_GROUP = "v3 (đã sửa, +'Query:')"
def v3_ckpt(tag, seed):
    return f"{SAVE_DIR}/stage3_ckpt_{tag}_seed{seed}.pt"

def time_docs_v3(net, D, M, n=200):
    idx = list(range(min(n, len(D))))
    ms_gpu = None
    if torch.cuda.is_available():
        batches = [pack_docs_grid(D, M, idx[i:i + 32]) for i in range(0, len(idx), 32)]
        with torch.no_grad():
            _, ms_gpu = sync_ms(lambda: [net.forward_docs(*b) for b in batches], len(idx))
        del batches
    net_cpu = copy.deepcopy(net).cpu().eval()
    batches = [pack_docs_grid(D, M, idx[i:i + 32], device="cpu") for i in range(0, len(idx), 32)]
    with torch.no_grad():
        t0 = time.perf_counter()
        for b in batches:
            net_cpu.forward_docs(*b)
        ms_cpu = (time.perf_counter() - t0) * 1000 / len(idx)
    return ms_gpu, ms_cpu

def run_v3(name, spec, data, teacher_tag=None):
    if name in RESULTS:
        print("✓ đã có:", name)
        return
    vi, en, ood, t0 = [], [], [], time.time()
    last = None
    for seed in SEEDS:
        print(f"\n>>> {name} | seed {seed}")
        teacher = None
        if teacher_tag is not None and spec["kd"]:
            teacher = GridMultiVectorAdapter(dim=EMB_DIM, rank=32, compress=False).to(DEVICE).eval()
            teacher.load_state_dict(torch.load(v3_ckpt(teacher_tag, seed), map_location=DEVICE))
        net = train_v3(spec, data, seed, teacher=teacher)
        if spec.get("ckpt"):
            torch.save(net.state_dict(), v3_ckpt(spec["ckpt"], seed))
        vi.append(eval_v3(net, Q_test, C_test, D_test, M_test, TEST_T)[0])
        if RUN_EN_CHECK:
            en.append(eval_v3(net, Q_en, C_en, D_en, M_en, EN_T)[0])
        if D_ood is not None:
            ood.append(eval_v3(net, Q_ood, C_ood, D_ood, M_ood, OOD_T)[0])
        last = net
        del teacher
        free()
    vpp = vec_per_page(encode_docs_v3(last, D_test, M_test))
    ms_gpu, ms_cpu = time_docs_v3(last, D_test, M_test)
    n_par = int(sum(p.numel() for p in last.parameters() if p.grad is not None))
    save_result(name, vi=vi, en=en or None, ood=ood or None, vec_per_page=vpp, embed_dim=EMB_DIM, ms_per_page=ms_gpu,
                ms_cpu_per_page=ms_cpu, device=f"{DEVICE_NAME}; CPU {ms_cpu:.2f} ms/trang", trained=True, group=V3_GROUP,
                n_seeds=len(SEEDS), n_params=n_par, spec=json.dumps(spec), train_min_per_seed=(time.time() - t0) / 60 / len(SEEDS))
    print(name, "| VI nDCG@5 theo seed:", [round(x, 1) for x in seed_scores(vi)])
    del last
    free()

# ---- V3-A: gộp 2D + anchor bao phủ, KHÔNG train (so trực tiếp với Hier×4 và Q-A của v2) ----
NAME_V3A = "V3-A: Grid 2×4 + anchor bao phủ (chưa train)"
if RUN_V3 and NAME_V3A not in RESULTS:
    net_a = GridMultiVectorAdapter(dim=EMB_DIM, compress=True, use_doc_adapter=False, use_subword_attn=False, use_query_gate=False).to(DEVICE).eval()
    r_vi, Dc = eval_v3(net_a, Q_test, C_test, D_test, M_test, TEST_T)
    r_en = eval_v3(net_a, Q_en, C_en, D_en, M_en, EN_T)[0] if RUN_EN_CHECK else None
    r_ood = eval_v3(net_a, Q_ood, C_ood, D_ood, M_ood, OOD_T)[0] if D_ood is not None else None
    ms_gpu, ms_cpu = time_docs_v3(net_a, D_test, M_test)
    save_result(NAME_V3A, vi=r_vi, en=r_en, ood=r_ood, vec_per_page=vec_per_page(Dc), embed_dim=EMB_DIM, ms_per_page=ms_gpu,
                ms_cpu_per_page=ms_cpu, device=f"{DEVICE_NAME}; CPU {ms_cpu:.2f} ms/trang", trained=False, group="zero-shot", n_seeds=1,
                per_page_vec=[int(len(x)) for x in Dc])
    del net_a, Dc
    free()

# ---- Hier cùng NGÂN SÁCH với v3 (số cụm của từng trang = số vector v3 trên trang đó) ----
from scipy.cluster.hierarchy import linkage, fcluster
from concurrent.futures import ThreadPoolExecutor

def hier_pool_to_k(E, k):
    """Giống hệt HierarchicalTokenPooler của colpali-engine (Ward trên 1 − cos, mean-pool + L2) nhưng số cụm = k."""
    x = E.float()
    n = x.shape[0]
    k = int(max(1, min(k, n)))
    if k >= n:
        return x.to(torch.bfloat16)
    lab = fcluster(linkage(1.0 - (x @ x.T).numpy(), metric="euclidean", method="ward"), t=k, criterion="maxclust") - 1
    lab_t = torch.as_tensor(lab)
    out = [F.normalize(x[lab_t == c].mean(0), dim=-1) for c in np.unique(lab)]
    return torch.stack(out).to(torch.bfloat16)

def hier_pool_budget(D, ks):
    with ThreadPoolExecutor(os.cpu_count()) as ex:
        return list(ex.map(lambda a: hier_pool_to_k(*a), zip(D, ks)))

NAME_HIER_B = "ZS: ColQwen2.5 + Hier cùng ngân sách v3 (+'Query:')"
if RUN_V3 and NAME_HIER_B not in RESULTS and NAME_V3A in RESULTS:
    ks = RESULTS[NAME_V3A]["per_page_vec"]
    t0 = time.perf_counter()
    Dh = hier_pool_budget(D_test, ks)
    ms_h = (time.perf_counter() - t0) * 1000 / len(D_test)
    # EN/OOD: số cụm theo cùng công thức cửa sổ của v3 (cận trên)
    Dh_en = hier_pool_budget(D_en, [v3_doc_count(m, len(E)) for m, E in zip(M_en, D_en)]) if RUN_EN_CHECK else None
    Dh_ood = hier_pool_budget(D_ood, [v3_doc_count(m, len(E)) for m, E in zip(M_ood, D_ood)]) if D_ood is not None else None
    r = eval_triplet({"vi": Q_test, "en": Q_en, "ood": Q_ood}, {"vi": Dh, "en": Dh_en, "ood": Dh_ood})
    save_result(NAME_HIER_B, vi=r["vi"], en=r.get("en"), ood=r.get("ood"), vec_per_page=vec_per_page(Dh), embed_dim=EMB_DIM,
                ms_per_page=ms_h, device=f"CPU x{os.cpu_count()}", trained=False, group="zero-shot", n_seeds=1)
    del Dh, Dh_en, Dh_ood

# ---- Các biến thể v3 (teacher V3-Full phải train trước) ----
V3_FULL = "V3-Full: Dual Adapter v3 không nén (V-Doc)"
V3_C = "V3-C: Grid 2×4 + Adapter v3 (+KD từ V3-Full, V-Doc)"
V3_SPECS = [
    (V3_FULL, dict(compress=False, doc=True, sub=True, gate=True, kd=False, ckpt="v3_full"), None, None),
    (V3_C, dict(compress=True, doc=True, sub=True, gate=True, kd=True, ckpt="v3_c"), None, "v3_full"),
    ("V3-C −KD: Grid 2×4 + Adapter v3 (chỉ CE, V-Doc)", dict(compress=True, doc=True, sub=True, gate=True, kd=False), None, None),
]
V3_ABLATIONS = [
    ("V3-C −gate (V-Doc)", dict(compress=True, doc=True, sub=True, gate=False, kd=True), None, "v3_full"),
    ("V3-C −subword attn (V-Doc)", dict(compress=True, doc=True, sub=False, gate=True, kd=True), None, "v3_full"),
    ("V3-C −doc adapter (chỉ gộp 2D + nhánh query, V-Doc)", dict(compress=True, doc=False, sub=True, gate=True, kd=True), None, "v3_full"),
]
V3_DATA = [
    ("V3-C + EN replay (V-Doc + colpali_train_set)", dict(compress=True, doc=True, sub=True, gate=True, kd=True), "replay_en", "v3_full"),
    ("V3-Full + EN replay (V-Doc + colpali_train_set)", dict(compress=False, doc=True, sub=True, gate=True, kd=False), "replay_en", None),
    ("V3-C + 5CD vi_image (V-Doc + 5CD-AI ảnh tiếng Việt)", dict(compress=True, doc=True, sub=True, gate=True, kd=True), "5cd_vi_image", "v3_full"),
]
if RUN_V3:
    DATA_VDOC = make_train_data()
    todo = V3_SPECS + (V3_ABLATIONS if RUN_ABLATIONS else [])
    extras = {"replay_en": EXTRA_REPLAY, "5cd_vi_image": EXTRA_5CDVI}
    for name, spec, extra_tag, teacher_tag in V3_DATA:
        if extras.get(extra_tag) is not None:
            todo.append((name, spec, extra_tag, teacher_tag))
    data_cache = {None: DATA_VDOC}
    for name, spec, extra_tag, teacher_tag in todo:
        if name in RESULTS:
            print("✓ đã có:", name)
            continue
        if extra_tag not in data_cache:
            data_cache[extra_tag] = make_train_data((extra_tag, extras[extra_tag]))
        run_v3(name, spec, data_cache[extra_tag], teacher_tag=teacher_tag)

## 11. Đối chứng: kiến trúc v2 nguyên bản + `'Query: '`

In [ ]:
# =====================================================================================
# Đối chứng: KIẾN TRÚC v2 GIỮ NGUYÊN, chỉ sửa giao thức query (+'Query:') và dùng chung hard negatives
#   -> tách bạch phần chênh lệch do sửa giao thức với phần do các sửa đổi kiến trúc/huấn luyện của v3
# =====================================================================================
V2R_GROUP = "v2 kiến trúc gốc (+'Query:')"
V2R_SPECS = {
    "V2-C (+'Query:'): DynMerge 1D ×4 + Adapter v2 (+KD, V-Doc)": ("dyn_merge", True, True, True, True, "stage3_ckpt_v2c_prefix"),
    "V2-Full (+'Query:'): Adapter v2 không nén (+KD, V-Doc)": ("none", True, True, True, True, None),
}
if RUN_V2_RERUN:
    for name, (d_mode, u_doc, u_sub, u_gate, u_kd, ck) in V2R_SPECS.items():
        if name in RESULTS:
            print("✓ đã có:", name)
            continue
        vi, en, ood, t0 = [], [], [], time.time()
        last = None
        for seed in SEEDS:
            print(f"\n>>> {name} | seed {seed}")
            net = train_qwen_adapter(Q_tr, D_tr, VDOC_TR_T, VDOC_TR_H, HARD_P, Q_val, D_val,
                                     d_mode, u_doc, u_sub, u_gate, u_kd, seed)
            if ck is not None:
                torch.save(net.state_dict(), f"{SAVE_DIR}/{ck}_seed{seed}.pt")
            r_vi, Dc = eval_adapter_ragged(net, Q_test, D_test, TEST_T)
            vi.append(r_vi)
            vpp = vec_per_page(Dc)
            del Dc
            if RUN_EN_CHECK:
                en.append(eval_adapter_ragged(net, Q_en, D_en, EN_T)[0])
            if D_ood is not None:
                ood.append(eval_adapter_ragged(net, Q_ood, D_ood, OOD_T)[0])
            if last is not None:
                del last
            last = net
            free()
        ms = time_doc_compression(last, D_test)
        save_result(name, vi=vi, en=en or None, ood=ood or None, vec_per_page=vpp, embed_dim=EMB_DIM, ms_per_page=ms,
                    device=DEVICE_NAME, trained=True, group=V2R_GROUP, n_seeds=len(SEEDS),
                    n_params=int(sum(p.numel() for p in last.parameters() if p.grad is not None)),
                    train_min_per_seed=(time.time() - t0) / 60 / len(SEEDS))
        print(name, "| VI nDCG@5 theo seed:", [round(x, 1) for x in seed_scores(vi)])
        del last
        free()

## 12. Phân tích Query Gate trên toàn bộ test (v3 so với v2)

In [ ]:
# =====================================================================================
# Phân tích Query Gate trên TOÀN BỘ 1.067 câu hỏi test (thay vì 5 câu như v2), chỉ tính trên token NỘI DUNG
#   - Pearson r(vị trí tương đối, trọng số) gộp mọi token; r trung bình theo từng câu
#   - Trọng số trung bình của từ đầu câu so với phần còn lại
#   - Token có trọng số trung bình cao/thấp nhất (xuất hiện ≥ 20 lần)
# =====================================================================================
def gate_report(label, gates_list, cm_list, ids_list):
    pos_all, w_all, per_q_r, first, rest = [], [], [], [], []
    tok_w = {}
    for g, cm, ids in zip(gates_list, cm_list, ids_list):
        g = np.asarray(g, dtype=float)
        cm = np.asarray(cm, dtype=bool)[:len(g)]
        idx = np.where(cm)[0]
        if len(idx) < 3:
            continue
        w = g[idx]
        rp = np.arange(len(idx)) / (len(idx) - 1)
        pos_all += rp.tolist(); w_all += w.tolist()
        if np.std(w) > 1e-9:
            per_q_r.append(np.corrcoef(rp, w)[0, 1])
        first.append(w[0]); rest.append(w[1:].mean())
        for j in idx:
            piece = PROC_Q.tokenizer.decode([ids[j]]).strip()
            if piece:
                tok_w.setdefault(piece, []).append(g[j])
    r_all = float(np.corrcoef(pos_all, w_all)[0, 1])
    stats = sorted(((np.mean(v), k, len(v)) for k, v in tok_w.items() if len(v) >= 20), reverse=True)
    print(f"\n[{label}] Pearson r(vị trí, trọng số) gộp = {r_all:+.3f} | r trung bình theo câu = {np.mean(per_q_r):+.3f} "
          f"| trọng số token đầu câu = {np.mean(first):.3f} vs phần còn lại = {np.mean(rest):.3f}")
    print("  Cao nhất :", [(k, round(m, 3), n) for m, k, n in stats[:12]])
    print("  Thấp nhất:", [(k, round(m, 3), n) for m, k, n in stats[-12:]])
    return {"r_all": r_all, "r_per_query": float(np.mean(per_q_r)), "first": float(np.mean(first)), "rest": float(np.mean(rest))}

GATE_SUMMARY = {}
if RUN_GATE_VIS:
    ck3 = v3_ckpt("v3_c", 0)
    if os.path.exists(ck3):
        net_g = GridMultiVectorAdapter(dim=EMB_DIM, rank=32, compress=True).to(DEVICE).eval()
        net_g.load_state_dict(torch.load(ck3, map_location=DEVICE))
        _, G3 = encode_queries_v3(net_g, Q_test, C_test)
        GATE_SUMMARY["v3"] = gate_report("V3-C seed 0 (gate chỉ trên token nội dung + phạt tương quan vị trí)", G3, C_test, IDS_test)
        del net_g
    ck2 = f"{SAVE_DIR}/stage3_ckpt_v2c_prefix_seed0.pt"
    if os.path.exists(ck2):
        net_g2 = DynamicMultiVectorAdapter(dim=EMB_DIM, rank=32, window_size=8, dropout=0.1, doc_mode="dyn_merge",
                                           use_doc_adapter=True, use_subword_attn=True, use_query_gate=True).to(DEVICE).eval()
        net_g2.load_state_dict(torch.load(ck2, map_location=DEVICE))
        G2 = []
        with torch.no_grad():
            for i in range(0, len(Q_test), 64):
                q_pad, q_mask = _pad_with_mask(Q_test[i:i + 64])
                _, g, _, _ = net_g2.forward_queries(q_pad.to(DEVICE), q_mask.to(DEVICE))
                G2 += [g[j].float().cpu()[q_mask[j]] for j in range(len(g))]
        GATE_SUMMARY["v2"] = gate_report("V2-C (+'Query:') seed 0 (gate gốc của v2)", G2, C_test, IDS_test)
        del net_g2
    free()

## 13. Bảng tổng hợp cuối cùng, slice analysis, Pareto & phát biểu kết quả tự sinh
- So sánh công bằng **trong cùng nhóm** (`Train V-Doc` ✓/✗). Dòng `[v2]` chỉ để đối chiếu (giao thức cũ).
- Cột `(Holm)` là p đã hiệu chỉnh cho toàn bộ họ so sánh. Cột `OOD` và `xa-train` là thước đo tổng quát hoá; V-Doc test là **cùng miền**.

In [ ]:
# =====================================================================================
# Bảng tổng hợp chuẩn hoá: Δ so với zero-shot (+'Query:') và Hier×4, randomization p + Holm (toàn bộ họ so sánh),
# CI95 bootstrap, EN DocVQA, OOD ViInfographicVQA, nửa test "xa train nhất" (nếu có)
# =====================================================================================
V2_IMPORT = {
    "SOTA: ColQwen2.5-3B (zero-shot)": ("[v2] ZS: ColQwen2.5-3B (không tiền tố)", False),
    "ColQwen2.5 + Hierarchical pooling ×4 (không train, CPU)": ("[v2] ZS: ColQwen2.5 + Hier×4 (không tiền tố)", False),
    "Q-A: ColQwen2.5 + DynMerge ×4 (chưa train, GPU)": ("[v2] Q-A: DynMerge ×4 chưa train", False),
    "Q-B: ColQwen2.5 + DynMerge ×4 + Adapter (Pure CE, V-Doc)": ("[v2] Q-B: DynMerge ×4 + Adapter (CE)", True),
    "Q-C: ColQwen2.5 + DynMerge ×4 + Adapter (+ KD & Anchor, V-Doc)": ("[v2] Q-C: DynMerge ×4 + Adapter (+KD)", True),
    "Q-Full: ColQwen2.5 (733 vec) + Dual Adapter (+ KD & Anchor, V-Doc)": ("[v2] Q-Full: Dual Adapter 733 vec", True),
    "Q-C-Mix: DynMerge ×4 + Adapter (+ KD, V-Doc + 5CD-AI)": ("[v2] Q-C-Mix (5CD cột image)", True),
    "Q-Full-Mix: ColQwen2.5 (733 vec) + Dual Adapter (+ KD, V-Doc + 5CD-AI)": ("[v2] Q-Full-Mix (5CD cột image)", True),
}

def import_v2_rows():
    out = {}
    for old, (new, trained) in V2_IMPORT.items():
        if old in RESULTS_V2:
            v = dict(RESULTS_V2[old])
            v.update(trained=trained, group="v2 (không tiền tố — tham chiếu)")
            out[new] = v
    return out

def _time_str(v):
    if v.get("encode_s_per_page") is not None:
        return f"{v['encode_s_per_page']:.2f} s/trang encode ({v.get('device', '')})"
    if v.get("ms_per_page") is not None:
        return f"{v['ms_per_page']:.2f} ms/trang ({v.get('device', '')})"
    return None

def build_master_table(R, order, ref_zs=REF_ZS, ref_hier=REF_HIER, far_mask=None):
    rows, p_cells = [], []
    ref_vi = {k: pq_mean(R[k]["vi"]) for k in (ref_zs, ref_hier) if k in R}
    ref_en = pq_mean(R[ref_zs]["en"]) if ref_zs in R and R[ref_zs].get("en") else None
    ref_ood = pq_mean(R[ref_zs]["ood"]) if ref_zs in R and R[ref_zs].get("ood") else None
    for name in [n for n in order if n in R]:
        v = R[name]
        vi = pq_mean(v["vi"])
        sc = seed_scores(v["vi"])
        vpp = v.get("vec_per_page")
        dim = int(v.get("embed_dim") or 128)
        lo, hi = boot_ci(vi)
        row = {
            "Mô hình": name, "Nhóm": v.get("group"), "Train V-Doc": "✓" if v.get("trained") else "✗", "Seeds": len(sc),
            "Tham số train": v.get("n_params"),
            "Vec/trang": round(float(vpp), 1) if vpp else None,
            "MB/1000 trang": round(MB_PER_1K(float(vpp), dim), 1) if vpp else None,
            "VI nDCG@5": f"{np.mean(sc):.1f}" + (f" ± {np.std(sc, ddof=1):.1f}" if len(sc) > 1 else ""),
            "VI R@1": round(float(np.mean(seed_scores(v["vi"], "R@1"))), 1),
            "VI R@5": round(float(np.mean(seed_scores(v["vi"], "R@5"))), 1),
            "CI95": f"[{lo:.1f}, {hi:.1f}]",
        }
        for ref, col in [(ref_zs, "ZS"), (ref_hier, "Hier×4")]:
            if ref in ref_vi and name != ref and len(ref_vi[ref]) == len(vi):
                row[f"Δ vs {col}"] = round(float((vi - ref_vi[ref]).mean()), 1)
                row[f"p vs {col}"] = paired_randomization_p(vi, ref_vi[ref])
                p_cells.append((len(rows), f"p vs {col}"))
        if v.get("en"):
            en = pq_mean(v["en"])
            row["EN nDCG@5"] = round(float(en.mean()), 1)
            if ref_en is not None and name != ref_zs and len(ref_en) == len(en):
                row["Δ EN"] = round(float((en - ref_en).mean()), 1)
                row["p EN"] = paired_randomization_p(en, ref_en)
                p_cells.append((len(rows), "p EN"))
        if v.get("ood"):
            od = pq_mean(v["ood"])
            row["OOD nDCG@5"] = round(float(od.mean()), 1)
            if ref_ood is not None and name != ref_zs and len(ref_ood) == len(od):
                row["Δ OOD"] = round(float((od - ref_ood).mean()), 1)
                row["p OOD"] = paired_randomization_p(od, ref_ood)
                p_cells.append((len(rows), "p OOD"))
        if far_mask is not None and ref_zs in ref_vi:
            row["VI xa-train nDCG@5"] = round(float(vi[far_mask].mean()), 1)
            if name != ref_zs:
                row["Δ xa-train"] = round(float((vi[far_mask] - ref_vi[ref_zs][far_mask]).mean()), 1)
        row["Thời gian"] = _time_str(v)
        rows.append(row)
    adj = holm_adjust([rows[i][c] for i, c in p_cells])
    for (i, c), a in zip(p_cells, adj):
        rows[i][c + " (Holm)"] = fmt_p(a)
        rows[i][c] = fmt_p(rows[i][c])
    df = pd.DataFrame(rows)
    lead = [c for c in ["Mô hình", "Nhóm", "Train V-Doc", "Seeds", "Tham số train", "Vec/trang", "MB/1000 trang",
                        "VI nDCG@5", "VI R@1", "VI R@5", "CI95", "Δ vs ZS", "p vs ZS", "p vs ZS (Holm)",
                        "Δ vs Hier×4", "p vs Hier×4", "p vs Hier×4 (Holm)", "EN nDCG@5", "Δ EN", "p EN", "p EN (Holm)",
                        "OOD nDCG@5", "Δ OOD", "p OOD", "p OOD (Holm)", "VI xa-train nDCG@5", "Δ xa-train", "Thời gian"] if c in df.columns]
    return df[lead]
# =====================================================================================
# BẢNG TỔNG HỢP CUỐI CÙNG: v2 (tham chiếu) + baseline fine-tune (notebook stage3_finetune_baselines) + v3
# =====================================================================================
RESULTS_A = json.load(open(RES_A_PATH, encoding="utf-8")) if os.path.exists(RES_A_PATH) else {}
if not RESULTS_A:
    print("⚠ Chưa có kết quả notebook baseline (stage3_finetune_baselines.ipynb) -> bảng thiếu các dòng FT-*.")
R_ALL = {**import_v2_rows(), **RESULTS_A, **RESULTS}
LORA_HIER_A = next((n for n in RESULTS_A if n.startswith("FT-LoRA") and "Hier" in n), None)
_v3_names = [n for n, *_ in V3_SPECS + V3_ABLATIONS + V3_DATA]
ORDER_B = ([REF_ZS_NOPFX, REF_ZS, REF_HIER, NAME_HIER_B, NAME_V3A, "[v2] Q-A: DynMerge ×4 chưa train"]
           + [n for n in RESULTS_A if n.startswith("FT-")]
           + ["[v2] Q-B: DynMerge ×4 + Adapter (CE)", "[v2] Q-C: DynMerge ×4 + Adapter (+KD)", "[v2] Q-Full: Dual Adapter 733 vec",
              "[v2] Q-C-Mix (5CD cột image)", "[v2] Q-Full-Mix (5CD cột image)"]
           + list(V2R_SPECS) + _v3_names)
df_final = build_master_table(R_ALL, ORDER_B, far_mask=FAR_MASK)
df_final.to_csv(f"{SAVE_DIR}/results_stage3_final_table.csv", index=False)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
display(df_final)

# ---- Slice analysis (dùng OCR Tesseract đã cache ở Giai đoạn 1 nếu có) ----
if RUN_SLICE_ANALYSIS and OCR_TEXTS is not None and len(OCR_TEXTS) == len(TEST_Q):
    ocr_lens = np.array([len(t.strip()) for t in OCR_TEXTS])
    has_cue = np.array([bool(re.search(r"\b(hình|bảng|ví dụ|biểu đồ|sơ đồ|câu|bài)\s*\d", q.lower())) for q in TEST_Q])
    slices = {
        f"Tất cả ({len(TEST_Q)})": np.ones(len(TEST_Q), dtype=bool),
        f"OCR ít (<200, n={(ocr_lens < 200).sum()})": ocr_lens < 200,
        f"OCR vừa (200–800, n={((ocr_lens >= 200) & (ocr_lens <= 800)).sum()})": (ocr_lens >= 200) & (ocr_lens <= 800),
        f"OCR dày (>800, n={(ocr_lens > 800).sum()})": ocr_lens > 800,
        f"Có nhãn số (n={has_cue.sum()})": has_cue,
        f"Không nhãn số (n={(~has_cue).sum()})": ~has_cue,
        f"Xa train (n={FAR_MASK.sum()})": FAR_MASK,
    }
    ref = pq_mean(R_ALL[REF_ZS]["vi"])
    slice_rows = []
    for m_name in [REF_ZS, REF_HIER, NAME_HIER_B, LORA_HIER_A,
                   "FT-E1: Linear-Q + Hier×4 (V-Doc, CE)", list(V2R_SPECS)[0], V3_C, V3_FULL]:
        if m_name is None or m_name not in R_ALL:
            continue
        sc = pq_mean(R_ALL[m_name]["vi"])
        row = {"Mô hình": m_name}
        for lab, mask in slices.items():
            lo, hi = boot_ci(sc[mask] - ref[mask]) if m_name != REF_ZS and mask.sum() > 1 else (np.nan, np.nan)
            row[lab] = f"{sc[mask].mean():.1f}" + ("" if m_name == REF_ZS else f" (Δ {np.mean(sc[mask] - ref[mask]):+.1f} [{lo:+.1f},{hi:+.1f}])")
        slice_rows.append(row)
    df_slice = pd.DataFrame(slice_rows)
    df_slice.to_csv(f"{SAVE_DIR}/slice_analysis_stage3.csv", index=False)
    print("\n--- Slice analysis (VI nDCG@5; Δ so với ZS +'Query:' kèm CI95 bootstrap của chênh lệch) ---")
    display(df_slice)

# ---- Pareto: bộ nhớ index vs VI nDCG@5 ----
if RUN_PLOT:
    fig, ax = plt.subplots(figsize=(10.2, 6.0), dpi=160)
    for name in ORDER_B:
        if name not in R_ALL or not R_ALL[name].get("vec_per_page"):
            continue
        v = R_ALL[name]
        mb = MB_PER_1K(float(v["vec_per_page"]), int(v.get("embed_dim") or 128))
        nd = float(np.mean(seed_scores(v["vi"])))
        trained = bool(v.get("trained"))
        col = "#d62828" if name.startswith("V3") else ("#457b9d" if name.startswith("FT-") else ("#999999" if name.startswith("[v2]") else "#1d3557"))
        ax.scatter(mb, nd, s=90 if name.startswith("V3") else 55, marker="o" if trained else "s", color=col, edgecolor="black", linewidth=0.5, zorder=3)
        ax.annotate(name.split(":")[0], (mb, nd), xytext=(5, 3), textcoords="offset points", fontsize=7)
    ax.set_xlabel("MB / 1.000 trang (bf16)")
    ax.set_ylabel("VI nDCG@5 (V-Doc test)")
    ax.set_title("Bộ nhớ index vs độ chính xác — tròn: có train V-Doc, vuông: không train", fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/pareto_stage3.png", dpi=200)
    plt.show()

# ---- Câu phát biểu kết quả TỰ SINH từ số liệu (để dán vào báo cáo; không tự diễn giải quá mức) ----
def _m(name, key="vi"):
    return float(np.mean(seed_scores(R_ALL[name][key]))) if name in R_ALL and R_ALL[name].get(key) else None

def _cmp(a, b, key="vi", mask=None):
    if a is None or b is None or a not in R_ALL or b not in R_ALL or not R_ALL[a].get(key) or not R_ALL[b].get(key):
        return None
    x, y = pq_mean(R_ALL[a][key]), pq_mean(R_ALL[b][key])
    if mask is not None:
        x, y = x[mask], y[mask]
    lo, hi = boot_ci(x - y)
    return float((x - y).mean()), lo, hi, paired_randomization_p(x, y)

print("\n================ PHÁT BIỂU KẾT QUẢ (tự sinh) ================")
z, zn = _m(REF_ZS), _m(REF_ZS_NOPFX)
if z is not None and zn is not None:
    en0, en1 = _m(REF_ZS_NOPFX, "en"), _m(REF_ZS, "en")
    en_txt = f" (EN {en0:.1f} → {en1:.1f})" if en0 is not None and en1 is not None else ""
    print(f"• Giao thức: thêm 'Query: ' đổi ColQwen2.5 zero-shot từ {zn:.1f} lên {z:.1f} VI nDCG@5{en_txt}. "
          "Mọi so sánh dưới đây dùng mốc có tiền tố.")
for a, b, lab in [(V3_C, REF_ZS, "V3-C so với ZS không nén"), (V3_C, NAME_HIER_B, "V3-C so với Hier cùng ngân sách"),
                  (V3_C, "FT-E1: Linear-Q + Hier×4 (V-Doc, CE)", "V3-C so với adapter tuyến tính + Hier×4 (cùng dữ liệu)"),
                  (V3_C, LORA_HIER_A, "V3-C so với LoRA + Hier×4"),
                  (V3_C, list(V2R_SPECS)[0], "V3-C so với kiến trúc v2 (+'Query:')"),
                  (V3_FULL, V3_C, "Chi phí nén (V3-Full − V3-C)"),
                  (NAME_V3A, NAME_HIER_B, "Bộ gộp v3 chưa train so với Hier cùng ngân sách")]:
    r = _cmp(a, b)
    if r is not None:
        print(f"• {lab}: Δ = {r[0]:+.1f} nDCG@5, CI95 [{r[1]:+.1f}, {r[2]:+.1f}], p = {fmt_p(r[3])} (chưa hiệu chỉnh; xem cột Holm trong bảng)")
for key, lab in [("en", "EN DocVQA"), ("ood", "OOD ViInfographicVQA")]:
    r = _cmp(V3_C, REF_ZS, key)
    if r is not None:
        print(f"• {lab}: V3-C − ZS = {r[0]:+.1f}, CI95 [{r[1]:+.1f}, {r[2]:+.1f}], p = {fmt_p(r[3])}")
r = _cmp(V3_C, REF_ZS, mask=FAR_MASK)
if r is not None:
    print(f"• Nửa test xa train nhất: V3-C − ZS = {r[0]:+.1f}, CI95 [{r[1]:+.1f}, {r[2]:+.1f}]")
if GATE_SUMMARY:
    print("• Gate:", {k: {kk: round(vv, 3) for kk, vv in v.items()} for k, v in GATE_SUMMARY.items()})
print("Lưu ý khi viết: test V-Doc là CÙNG MIỀN với train (sách giáo khoa, câu hỏi do LLM sinh); "
      "chỉ OOD và nửa 'xa train' phản ánh khả năng tổng quát hoá.")